In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib

from scipy import interpolate
from scipy.stats import norm
from scipy.optimize import curve_fit
from scipy.linalg import svd, solve, det
from scipy.signal import savgol_filter

from exocrires import info
from exocrires import plotMatrix 

from astropy.io import fits

import glob
import tqdm

from excalibuhr.data import DETECTOR
from matplotlib.colors import SymLogNorm

path = '/data2/peng/project_DH_Tau_B/'

### 0. Load the data

In [ ]:
night='2023-01-01'

extr2d_A = DETECTOR(filename=f'/data2/peng/{night}/out/combined/Extr2D_PRIMARY__COMBINED_VDHTauA+Bcenter_K2166_NOD_A.npz')
extr2d_B = DETECTOR(filename=f'/data2/peng/{night}/out/combined/Extr2D_PRIMARY__COMBINED_VDHTauA+Bcenter_K2166_NOD_B.npz')

data = fits.open(f'/data2/peng/{night}/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
hdu_wave = data[1]

flux2d_A = extr2d_A.flux
flux2d_B = extr2d_B.flux

flux2d_A = np.array(flux2d_A)

flux2d_B_pad = np.zeros(shape=flux2d_A.shape)

for i in range(flux2d_B_pad.shape[0]):

    for j in range (flux2d_B_pad.shape[1]):

        arr=flux2d_B[i][j].shape[0]

        flux2d_B_pad[i][j][0:arr] = flux2d_B[i][j]


variance2d_A = extr2d_A.var
variance2d_B = extr2d_B.var

psf2d_A = extr2d_A.psf
psf2d_B = extr2d_B.psf

Nodding = ['A', 'B']
i=0
aper_comp=int(np.ceil(60-(2.3/0.059)))
for flux2d in [flux2d_A, flux2d_B]:

    for order in range(0,7):

        fig, axes = plt.subplots(3, 3, figsize=(30, 10))
        fig.suptitle('Nodding %s, Order  %s' %(Nodding[i], (23 + order)), fontsize=16)


        for detector in range(0,3):

            #plotMatrix.plotMatrix(flux2d_A[0][order], [0,2048], np.arange(0, 20, 1), 'Wavelength Axis', 'Spatial Axis', planet_posi=10, scale='log')
            wave=hdu_wave.data[detector, order]

            ax = axes[detector,0]
            im = ax.imshow(flux2d[detector][order], aspect='auto', norm=SymLogNorm(5, 500))

            # Set x-axis label and ticks
            x_ticks = np.arange(0, wave.size, step=300)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(wave[x_ticks].round(2), fontsize=8)

            if detector == 2:
                ax.set_xlabel('Wavelength (nm)')

            cbar = plt.colorbar(im, ax=ax)
            #cbar.set_ticks([-10, 0, 10])
            #cbar.set_ticklabels([-10,0,10])
            
            ax2 = axes[detector,1]
            ax2.plot(flux2d[detector][order][aper_comp-1])
            ax2.set_ylim(-10, 30)
            ax2.hlines(y=0, xmin=0, xmax=flux2d[detector][order].shape[1], linestyle='--', colors='grey')

            ax2.set_xticks(ax.get_xticks())
            ax2.set_xticklabels(ax.get_xticklabels())
            ax2.set_ylabel('Spatial axis')


            if detector == 2:
                ax2.set_xlabel('Wavelength (nm)')

            ax3 = axes[detector,2]
            spatial_curve=np.trapz(flux2d[detector][order][:,], x=wave, axis=1)
            ax3.plot(spatial_curve)

            ax3.vlines(x=60, ymin=0, ymax=np.nanmax(spatial_curve), linestyles='--', colors='grey')
            ax3.vlines(x=aper_comp-1, ymin=0, ymax=np.nanmax(spatial_curve), linestyles='--', colors='grey')


            ax3.set_yscale('symlog')
            ax3.set_ylim(np.nanmax([np.nanmin(spatial_curve),5]), np.nanmax(spatial_curve)*1.5)
            ax3.set_ylabel('Flux')

            if detector == 2:
                ax3.set_xlabel('Spatial axis')
                

        
        plt.show()
    i+=1
    #plotMatrix.plotMatrix(flux2d_A[0][order]+flux2d_B[0][order], [0,2048], np.arange(0, 40, 1), 'Wavelength Axis', 'Spatial Axis', planet_posi=None, scale='log')
    #plt.title('Nodding A + B, Order  %s'%(29-order))

#### 0.1 Define the functions and classes for post-processing

##### 0.1.1 Functions following Landman et al. 2024

In [ ]:
import numpy as np
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.linalg import solve

import scipy.ndimage
import scipy.constants

from scipy import interpolate
from astropy.io import fits
from PyAstronomy import pyasl
from exotic_ld import StellarLimbDarkening

import matplotlib.pyplot as plt
import tqdm

import glob




import os

class Spectra_Preparator:
    def __init__(self, path, folder_name, T, G, Z):
        self.path = path
        self.folder_name = folder_name
        self.path_a = os.path.join(path, folder_name)
        self.T = T
        self.G = G
        self.Z = Z
        self.ld_coefficients = None

    def generate_synthetic_stellar_models(self, model_spec, n_mus, factor):
        # Generate I(lambda, mu).
        n_wvs = len(model_spec)
        mus = np.linspace(0, 1, n_mus)
        stellar_intensity = []

        for mu in mus:
            stellar_intensity.append(model_spec*factor)

        return mus, np.array(stellar_intensity).T

    def compute_limb_darkening(self, wavelength_range, throughput_files, Mode='phoenix', flux=None, wave=None):
        """
        Compute limb darkening coefficients for different bands.
        """
        ld_coeffs = np.zeros((len(wavelength_range), 1))

        for idx, (lam_range, throughput_file) in enumerate(zip(wavelength_range, throughput_files)):
            throughput = np.loadtxt(throughput_file)
            wave_throughput, frac_throughput = throughput[:, 0], throughput[:, 1]

            if Mode == 'phoenix':
                sld = StellarLimbDarkening(
                    ld_model=Mode,
                    ld_data_path=os.path.join(self.path, 'ld_data'),
                    M_H=self.Z,
                    Teff=self.T,
                    logg=self.G
                )

                cs = sld.compute_linear_ld_coeffs(
                    mode='custom',
                    wavelength_range=[lam_range[0][0]*10, lam_range[-1][-1]*10],
                    custom_wavelengths=wave_throughput,
                    custom_throughput=frac_throughput
                )

            elif Mode =='custom':
                mus, intensities = self.generate_synthetic_stellar_models(flux, 70, factor=1e-8)
                
                sld=StellarLimbDarkening(
                    ld_model=Mode, 
                    ld_data_path=os.path.join(self.path, 'ld_data'), \
                    custom_wavelengths=wave, 
                    custom_stellar_model=intensities, 
                    custom_mus=mus
                )
                


                cs=sld.compute_linear_ld_coeffs(
                    mode='custom', wavelength_range=[lam_range[0][0]*10, lam_range[-1][-1]*10],   
                    custom_wavelengths=wave_throughput, 
                    custom_throughput=frac_throughput
                )




            ld_coeffs[idx] = np.float16(cs)

        self.ld_coefficients = ld_coeffs
        return ld_coeffs

    def load_phoenix_spectrum(self):
        """
        Load high resolution PHOENIX model spectrum.
        """
        flux = fits.open(os.path.join(
            self.path_a,
            f"lte0{self.T}-{self.G}0{self.Z}.PHOENIX-ACES-AGSS-COND-2011-HiRes.fits"
        ))[0].data

        wave = fits.open(os.path.join(
            self.path_a,
            "WAVE_PHOENIX-ACES-AGSS-COND-2011.fits"
        ))[0].data

        return wave, flux

    def gaussian_broaden(self, wavelength, flux, R):
        log_wavelength = np.log(wavelength)
        fwhm_log = 1 / R
        sigma_log = fwhm_log / 2.355
        broadened_flux = scipy.ndimage.gaussian_filter1d(
            flux,
            sigma=sigma_log / (log_wavelength[1] - log_wavelength[0])
        )
        return broadened_flux

    def resolution_sample(self, ave_R, wavelength):
        lam_range = np.max(wavelength) - np.min(wavelength)
        lam_mid = (np.max(wavelength) + np.min(wavelength)) / 2
        N = ave_R * (lam_range / lam_mid) + 1
        return int(N)

    def apply_broadening(self, wave, flux, lam_range, ld_coeff, R=1e5, rotation=5e0, reu=500, step_size=1e-11):

        if (wave[1]-wave[0]) != (wave[-1]-wave[-2]):

            print ('Sample grid is not equal. Use pyasl.equidistantInterpolation to produce equidistantly sampled data. ')

            
            wave_new, flux_new = pyasl.equidistantInterpolation(wave, flux, step_size)

    
        mask =  ( (wave_new >= (lam_range[0][0]-reu)*1e-9) & (wave_new <= (lam_range[-1][-1]+reu)*1e-9) )
        rflux = pyasl.fastRotBroad(
            wvl=wave_new[mask],
            flux=flux_new[mask],
            epsilon=ld_coeff,
            vsini=rotation
        )
        broadened_flux = self.gaussian_broaden(wave_new[mask], rflux, R)

        N = self.resolution_sample(R*2, wave_new[mask])
        wave_grid = np.linspace(np.min(wave_new[mask]), np.max(wave_new[mask]), N)
        inter_flux = np.interp(wave_grid, wave_new[mask], broadened_flux)

        return wave_grid, inter_flux


    def radial_velocity_shift(self, wave, flux, velocity):
        """
        Apply radial velocity shift in km/s.
        """
        shifted_wave = wave * (1 + (velocity * 1e3 / scipy.constants.c))
        return shifted_wave, flux
    
    def spectrum_cutter(self, lam_range, wave, flux, saving_file=True, saving_path = None, filenames = None, ignore_detector=True, Wave_solution=None):
        
        if ignore_detector == True:
            print ('The division into 3 detectros is ignored. The spectrum is only cut into different spectral orders.')
            n=0
            for i in lam_range:

                mask = ( ( wave >= i[0]*1e-9) & (wave <= i[1]*1e-9) )

                wave_cut = wave[mask]
                flux_cut = flux[mask]

                if saving_file == True:
                
                    filename = saving_path + filenames[n]

                    self.save_spectrum(filename, wave_cut, flux_cut)

                    n+=1
            
            print ('All spectra are saved in %s'%saving_path)

        if ignore_detector == False:

            spec_matrix = self.model_matrix_intepolater(wave, flux, Wave_solution)

            if saving_file == True:
                
                filename = '%s'%saving_path + '%s'%filenames

                np.save(filename, spec_matrix)

            print ('The spectral matrix is saved in %s with the shape of %s'%(saving_path, spec_matrix.shape))

            return (spec_matrix)


    def model_matrix_intepolater (self, wave_model, flux_model, wave_solution):

        spec_interp_matrix = np.zeros( shape=wave_solution.shape )

        for det in range(wave_solution.shape[0]):

            for order in range(wave_solution.shape[1]):

                wave_cut = wave_solution[det][order]

                model_mask = np.array((wave_model>=wave_cut.min())&(wave_model<=wave_cut.max()))

                spec_interp = np.interp(wave_cut, wave_model[model_mask], flux_model[model_mask])

            
                spec_interp_matrix[det][order]=spec_interp
        
        return (spec_interp_matrix)




    
    def save_spectrum(self, filename, wave, flux):
        spec = np.array([wave, flux])
        np.savetxt(filename, spec)

        return (spec)





class Spectra_fitting:
    """
    Assumes self.spectra has shape (nDet, nOrder, n_spatial, nPix)
    """

    def __init__(self, spectra2D, transmission_matrix=None, window_length=None, polyorder=None, detector_number=None):
        self.spectra = spectra2D
        self.transmission_matrix = transmission_matrix
        self.window_length = window_length
        self.polyorder = polyorder
        self.detector_number = detector_number

    def Mask_spec(self, atm_threshold, sigma_threshold, transmission_atmos=None, Mask_bad_pxiel=True):
        """
        transmission_atmos assumed to have shape compatible with spectra:
        e.g. (nDet, nOrder, nPix) or (nDet, nOrder, n_spatial, nPix) — here we assume (nDet, nOrder, nPix).
        Mask pixels below atm_threshold and outliers beyond sigma_threshold*(std).
        """
        nDet, nOrder, n_spatial, nPix = self.spectra.shape

        if transmission_atmos == None:
            transmission_atmos = self.transmission_matrix #Using the default transmission matrix

        # If transmission_atmos shape includes spatial axis, adapt:
        # Common use-case: transmission has shape (nDet, nOrder, nPix)
        # We'll treat transmission_atmos[iDet, iOrder, :] as the transmission per pixel.
        for iDet in range(nDet):
            for iOrder in range(nOrder):
                trans = transmission_atmos[iDet, iOrder]  # shape (nPix,) ideally

                # Build atmospheric mask for pixels below threshold
                low_atmo_mask = (trans <= atm_threshold)

                # For each spatial pixel compute mean/std across pixels or across spatial axis?
                # I assume we want to flag bad pixels per spatial row: find mean and std across wavelength axis for each pixel
                # mean over spatial axis -> shape (nPix,)
                if Mask_bad_pxiel == True:
                    mean_flux = np.nanmean(self.spectra[iDet, iOrder, :, :], axis=1)
                    std_flux = np.nanstd(self.spectra[iDet, iOrder, :, :], axis=1)

                    # For each spatial position, mask outliers along pixel axis
                    for isp in range(n_spatial):
                        flux_row = self.spectra[iDet, iOrder, isp, :]
                        # bad pixels where abs(flux - mean) > sigma_threshold * std
                        bad_pixel_mask = np.abs(flux_row - mean_flux[isp]) > (sigma_threshold * (std_flux[isp] + 1e-12))
                        # combine with atmosphere mask
                        combined_mask = bad_pixel_mask | low_atmo_mask
                        flux_row[combined_mask] = np.nan
                        self.spectra[iDet, iOrder, isp, :] = flux_row
                else:
                    for isp in range(n_spatial):
                        flux_row = self.spectra[iDet, iOrder, isp, :]
                        flux_row[low_atmo_mask] = np.nan
                        self.spectra[iDet, iOrder, isp, :] = flux_row

        return self.spectra

    def stellar_master_spec(self, wave, cut_off, center=None):
        """
        Compute a master stellar spectrum by integrating spatial pixels around center.
        By default it uses detector=det and order=order to find the spatial center.
        wave is the wavelength array matching last axis (length nPix).
        """
        nDet, nOrder, n_spatial, nPix = self.spectra.shape
        spec_master_matrix = np.zeros(shape=wave.shape)

        for iDet in range(nDet):
            for iOrder in range(nOrder):
                if center is None:
                    # integrate flux across wavelength for each spatial pixel to find center
                    # mask=np.isfinite(self.spectra[iDet, iOrder])

                    spatial_curve = np.trapz(np.nan_to_num(self.spectra[iDet, iOrder], nan=0.0), x=wave[iDet,iOrder], axis=1)  # shape (n_spatial,)
                    
                    if np.all(np.isnan(spatial_curve)):
                        print(f"Warning: all NaN curve in Det {iDet}, Order {iOrder}")
                        continue

                    center = int(np.nanargmax(spatial_curve))

                    print('Center: pixel %s' % center)

                i0 = max(0, center - cut_off)
                i1 = min(n_spatial, center + cut_off + 1)
                spec_master_matrix[iDet, iOrder, :] = np.trapz(np.nan_to_num(self.spectra[iDet, iOrder, i0:i1, :], nan=0.0), axis=0)  # integrate over selected spatial px
        

        return spec_master_matrix

    def fit_star(self, c, master, observation, window_length, polyorder):

        window_length=self.window_length
        polyorder=self.polyorder

        smoothed_master = savgol_filter(master, window_length, polyorder)
        smoothed_obs = savgol_filter(observation, window_length, polyorder)
        # avoid division by zero
        denom = smoothed_master.copy()
        denom[denom == 0] = np.nan
        alpha_star = smoothed_obs / denom
        stellar_real_obs = c * alpha_star * master
        stellar_model_contribution = alpha_star * master
        return (stellar_real_obs, stellar_model_contribution, alpha_star)

    def fit_planet(self, c, master, transmission, spec_planet, window_length, polyorder):

        window_length=self.window_length
        polyorder=self.polyorder

        smoothed_master = savgol_filter(master, window_length, polyorder)
        smoothed_planet = savgol_filter(transmission * spec_planet, window_length, polyorder)
        denom = smoothed_master.copy()
        denom[denom == 0] = np.nan
        alpha_planet = smoothed_planet / denom
        planet_obs = c * transmission * spec_planet
        planet_leakage = c * master * alpha_planet
        planet_real_obs = planet_obs - planet_leakage
        planet_model_contribution = transmission * spec_planet - alpha_planet * master
        return (planet_real_obs, planet_model_contribution, planet_obs, planet_leakage, alpha_planet)

    def multiplicative_factor_fit(self, planet_model_spectrum, master=None, window_len=None, Polyorder=None):
        """
        planet_model_spectrum shape expected: (nDet, nOrder, n_spatial?, nPix) or (nDet, nOrder, nPix)
        Here I assume planet_model_spectrum has shape (nDet, nOrder, nPix) — per det/order a single model
        star_master is 1D of length nPix (or if per det/order, match shapes)
        Returns: fitted planet and star matrices matching self.spectra dimensions.
        """
        self.window_length = window_len
        self.polyorder = Polyorder

        nDet, nOrder, n_spatial, nPix = self.spectra.shape

        popt_list = np.zeros((nDet, nOrder, n_spatial, 2))  # store cs and cp per (det,order,spatial)
        fit_matrix_planet = np.zeros_like(self.spectra)
        fit_matrix_star = np.zeros_like(self.spectra)
        fit_alpha_star = np.zeros_like(self.spectra)
        fit_alpha_planet = np.zeros_like(self.spectra)
        fit_model = np.zeros((2, nDet, nOrder, n_spatial, nPix))

        # If star_master is 1D, broadcast to (nPix)
        for iDet in range(nDet):
            for iOrder in range(nOrder):
                # Determine planet model slice for this det,order
                # Accept the only shapes: (nDet,nOrder,nPix)
                if planet_model_spectrum.ndim == 3:
                    planet_model_per_spatial = planet_model_spectrum[iDet, iOrder, :]  # (n_spatial, nPix)
                else:
                    print ('The only shape accpected is (nDet,nOrder,nPix).')

                # master for this det/order
                if master.ndim == 1:
                    master_vec = master
                elif master.ndim == 3:
                    master_vec = master[iDet, iOrder]
                else:
                    master_vec = master  # fallback

                mask_master = np.array(master_vec != 0.)

                for isp in range(n_spatial):
                    obs = self.spectra[iDet, iOrder, isp, :]
                    mask_obs = np.isfinite(obs)
                    mask = mask_master & mask_obs

                    if np.sum(mask) < 10:
                        # not enough points to fit, leave NaNs
                        popt_list[iDet, iOrder, isp, :] = np.nan
                        continue

                    def wrap_model(master_wrap, cs, cp):
                        star_comp = self.fit_star(cs, master_wrap, obs[mask], self.window_length, self.polyorder)[0]
                        pl_comp = self.fit_planet(cp, master_wrap, self.transmission_matrix[iDet, iOrder][mask], planet_model_per_spatial[mask], self.window_length, self.polyorder)[0]
                        return star_comp + pl_comp

                    try:
                        #popt, pcov = curve_fit(lambda m, cs, cp: wrap_model(m, cs, cp),
                                               #master_vec[mask], obs[mask])

                        popt, pcov = curve_fit(wrap_model, master_vec[mask], obs[mask])
                        
                    except Exception as e:
                        # fitting failed; store NaNs
                        popt = [np.nan, np.nan]
                        pcov = None

                    popt_list[iDet, iOrder, isp, :] = popt

                    # store fitted contributions
                    star_fit = self.fit_star(popt[0], master_vec[mask], obs[mask], self.window_length, self.polyorder)
                    planet_fit = self.fit_planet(popt[1], master_vec[mask], self.transmission_matrix[iDet, iOrder][mask], planet_model_per_spatial[mask], self.window_length, self.polyorder)

                    fit_matrix_planet[iDet, iOrder, isp, :][mask] = planet_fit[0]
                    fit_matrix_star[iDet, iOrder, isp, :][mask] = star_fit[0]
                    fit_alpha_star[iDet, iOrder, isp, :][mask] = star_fit[-1]
                    fit_alpha_planet[iDet, iOrder, isp, :][mask] = planet_fit[-1]
                    fit_model[0, iDet, iOrder, isp, :][mask] = star_fit[1]
                    fit_model[1, iDet, iOrder, isp, :][mask] = planet_fit[1]

        return (fit_matrix_planet, fit_matrix_star, fit_alpha_star, fit_alpha_planet, fit_model, popt_list)
    


class processor_cross_correlation:
    def __init__(self, wMod, fMod, wlen, cube, nOrder, n_spatial, nDet):
        """
        wMod, fMod : 1D model wavelength and flux (same length)
        wlen : observed wavelength grid with shape (nDet, nOrder, nPix)
        cube : observed cube shape (nDet, nOrder, n_spatial, nPix)
        """
        self.wMod = wMod
        self.fMod = fMod
        self.wlen = wlen
        self.cube = cube
        self.n_spatial = n_spatial
        self.nOrder = nOrder
        self.nDet = nDet

    def xcorr(self, f, g):
        """
        Normalized cross-correlation between 1D arrays f and g (no in-place edits).
        Returns NaN if varf or varg is zero.
        """
        f = np.asarray(f, dtype=float)
        g = np.asarray(g, dtype=float)
        nx = len(f)
        if nx == 0:
            return np.nan
        I = np.ones(nx)
        f_mean_sub = f - (np.dot(f, I) / nx)
        g_mean_sub = g - (np.dot(g, I) / nx)
        R = np.dot(f_mean_sub, g_mean_sub) / nx
        varf = np.dot(f_mean_sub, f_mean_sub) / nx
        varg = np.dot(g_mean_sub, g_mean_sub) / nx
        denom = np.sqrt(varf * varg)
        if denom == 0 or np.isnan(denom):
            return np.nan
        return R / denom

    def get_cc_grid(self, rvlag, ncc):
        """
        Compute CCF cube with shape (nDet, nOrder, n_spatial, ncc)
        """
        ccf = np.zeros((self.nDet, self.nOrder, self.n_spatial, ncc))
        coef_spline = interpolate.splrep(self.wMod, self.fMod, s=0.0)

        for irv, rv in enumerate(rvlag):
            beta = rv / 2.998e5
            # Relativistic shift: For each observed wavelength grid
            # wShift will be same shape as self.wlen (nDet,nOrder,nPix)
            wShift = self.wlen * np.sqrt((1.0 - beta) / (1.0 + beta))
            # Evaluate model flux at the shifted observed wavelengths
            # splev supports array input and will return an array same shape as wShift
            intMod = interpolate.splev(wShift, coef_spline, der=0)  # shape (nDet,nOrder,nPix)

            for iDet in range(self.nDet):
                for iOrder in range(self.nOrder):
                    for iObs in range(self.n_spatial):
                        obs = self.cube[iDet, iOrder, iObs, :]
                        model_row = intMod[iDet, iOrder, :]
                        # mask NaNs to avoid propagating
                        mask = np.isfinite(obs) & np.isfinite(model_row)
                        if np.sum(mask) < 5:
                            ccf[iDet, iOrder, iObs, irv] = np.nan
                        else:
                            ccf[iDet, iOrder, iObs, irv] = self.xcorr(obs[mask], model_row[mask])

        self.rvlag = rvlag
        self.ncc = ncc
        return ccf

    def ccf_tot(self, rvlag, ncc, plot=True, subtract_continuum='med_flux', normalization='median subtracted',
                v_sys=None, clean_grids=None, po=None, central_pix=None, spatial_pix=None):
        """
        Returns ccf_Sum (shape (n_spatial, ncc) summed across dets and orders),
                ccf_SNR (same shape)
        """

        # compute medFlux per det,order,pix by taking median over spatial axis
        medFlux = np.zeros((self.nDet, self.nOrder, self.wlen.shape[-1]))
        for iDet in range(self.nDet):
            for iOrder in range(self.nOrder):
                # cube slice shape (n_spatial, nPix), median over spatial axis -> (nPix,)
                medFlux[iDet, iOrder, :] = np.nanmedian(self.cube[iDet, iOrder, :, :], axis=0)

        # ccfWeight could be sum across dets then across orders if desired; keep it optional
        ccfWeight = np.sum(medFlux, axis=(0, 1))  # shape (nPix,)
        if np.nansum(ccfWeight) != 0:
            ccfWeight = ccfWeight / np.nansum(ccfWeight)

        ccf = self.get_cc_grid(rvlag, ncc)  # shape (nDet, nOrder, n_spatial, ncc)

        # Sum across detectors and orders to get per spatial x rv matrix
        ccf_Sum = np.nansum(ccf, axis=(0, 1))  # shape (n_spatial, ncc)

        # Normalization options
        if normalization == 'median':
            denom = np.nanmedian(ccf_Sum)
            if denom != 0:
                ccf_Sum = (ccf_Sum - denom) / denom
        elif normalization == 'max':
            mval = np.nanmax(ccf_Sum)
            if mval != 0:
                ccf_Sum = ccf_Sum / mval
        elif normalization == 'median subtracted':
            for iObs in range(self.n_spatial):
                med = np.nanmedian(ccf_Sum[iObs, :])
                ccf_Sum[iObs, :] -= med

        # optional plotting
        if plot:
            try:
                plotMatrix.plotMatrix(ccf_Sum, rvlag, np.arange(0, self.n_spatial, 1),
                                      'Radial velocity (km/s)', 'spatial axis', stretch=True,
                                      planet_posi=spatial_pix, scale='log')
            except Exception:
                plt.imshow(ccf_Sum, aspect='auto', origin='lower', extent=(rvlag[0], rvlag[-1], 0, self.n_spatial))
                plt.colorbar()
            if central_pix is not None:
                plt.hlines(y=central_pix, xmin=rvlag[0], xmax=rvlag[-1], ls='--', colors='lightgray')
            if v_sys is not None:
                plt.vlines(x=v_sys, ymin=0, ymax=(self.n_spatial - 1), ls='--', colors='lightgray')
            plt.show()

        # compute SNR by building a 'clean' grid (exclude RV ranges where signal sits)
        if clean_grids is None:
            # fallback: compute std across entire rv axis
            std_ccf = np.nanstd(ccf_Sum)
        else:
            # clean_grids expected like [(rv_idx0, rv_idx1), (rv_idx2, rv_idx3)]
            g0, g1 = int(clean_grids[0][0]), int(clean_grids[0][1])
            g2, g3 = int(clean_grids[1][0]), int(clean_grids[1][1])
            ccf_clean_map = np.concatenate((ccf_Sum[:, g0:g1], ccf_Sum[:, g2:g3]), axis=1)
            std_ccf = np.nanstd(ccf_clean_map)

        ccf_SNR = np.zeros_like(ccf_Sum)
        if std_ccf == 0:
            ccf_SNR[:] = np.nan
        else:
            ccf_SNR = ccf_Sum / std_ccf

        if plot:
            try:
                plotMatrix.plotMatrix(ccf_SNR, rvlag, np.arange(0, self.n_spatial, 1),
                                      'Radial velocity (km/s)', 'spatial axis', stretch=True,
                                      planet_posi=spatial_pix)
            except Exception:
                plt.imshow(ccf_SNR, aspect='auto', origin='lower', extent=(rvlag[0], rvlag[-1], 0, self.n_spatial))
                plt.colorbar()
            if central_pix is not None:
                plt.hlines(y=central_pix, xmin=rvlag[0], xmax=rvlag[-1], ls='--', colors='lightgray')
            if v_sys is not None:
                plt.vlines(x=v_sys, ymin=0, ymax=(self.n_spatial - 1), ls='--', colors='lightgray')
            plt.title('Cross-correlated SNR')
            plt.show()

        return (ccf_Sum, ccf_SNR)



class processor_likelihood_map:
    def __init__(self):
        pass

    def covariance_calculator(self, Fit_Matrix, ABBA_series, N_exposures):
        fit_matrix_single = Fit_Matrix / N_exposures
        fit_matrix_ex = np.expand_dims(fit_matrix_single, axis=1)
        fit_matrix_ex = np.repeat(fit_matrix_ex, repeats=N_exposures, axis=1)
        resi = ABBA_series - fit_matrix_ex
        var_resi = np.var(resi, axis=1)
        return (var_resi, resi)

    def likelihood_calculator(self, d=None, M=None, Sigma_0_flat=None, log_sigma_0=True, prior_psi=1.0, gamma=2):
        Sigma_0 = np.diagflat(Sigma_0_flat)
        # avoid log of zero
        safe_sigma_flat = np.array(Sigma_0_flat) + 1e-12
        log_Sigma_0 = np.diagflat(np.log(safe_sigma_flat))
        Sigma_0_inv = np.linalg.inv(Sigma_0)

        # Matrix arithmetic
        MT_Sigma0_inv_M = M.T @ Sigma_0_inv @ M
        MT_Sigma0_inv_d = d.T @ Sigma_0_inv @ M

        # Solve for c_hat: shape (Nc,)
        try:
            c_hat_T = solve(MT_Sigma0_inv_M, MT_Sigma0_inv_d)
            c_hat = c_hat_T.T
        except Exception as e:
            # fallback if singular
            c_hat = np.linalg.lstsq(MT_Sigma0_inv_M, MT_Sigma0_inv_d, rcond=None)[0].T

        residual = d - M @ c_hat
        chi_0_squared = float(residual.T @ Sigma_0_inv @ residual)

        # determinant handling (use slogdet for stability)
        sign_det, logdet = np.linalg.slogdet(MT_Sigma0_inv_M)
        if sign_det <= 0:
            # singular or negative determinant; fall back to small value
            logdet = np.log(np.abs(np.linalg.det(MT_Sigma0_inv_M) + 1e-30))

        Nd = len(d)
        Nc = M.shape[1]

        if not log_sigma_0:
            det_sigma0 = np.linalg.det(Sigma_0)
            log_likelihood = (np.log(prior_psi) - 0.5 * np.log(det_sigma0 * max(np.linalg.det(MT_Sigma0_inv_M), 1e-30))
                              + ((Nd - Nc + gamma - 1) / 2) * np.log(1.0 / max(chi_0_squared, 1e-30)))
        else:
            sum_log_sigma0 = np.sum(np.log(safe_sigma_flat))
            log_likelihood = (np.log(prior_psi) - 0.5 * (sum_log_sigma0 - logdet)
                              + ((Nd - Nc + gamma - 1) / 2) * np.log(1.0 / max(chi_0_squared, 1e-30)))
        return log_likelihood

    def likelihood_map(self, observation_matrix, model_matrix, n_vgrids, sigma_matrix, prior, matrix_components=2, log_sigma=True):
        """
        observation_matrix: shape (nObs, nPix)
        model_matrix: either shape (nVgrid, n_components, nObs, nPix) or (nVgrid, nObs, n_components, nPix)
        sigma_matrix: shape (nObs, nPix) variances
        Returns log_likelihood_mapp shape (nObs, nVgrid)
        """
        map_size = (observation_matrix.shape[0], n_vgrids)
        log_likelihood_mapp = np.zeros(shape=map_size)

        for v in tqdm.tqdm(range(model_matrix.shape[0])):
            for i in range(observation_matrix.shape[0]):
                variance_flat = sigma_matrix[i].flatten()
                mask_finite = np.isfinite(observation_matrix[i])
                mask_finite_var = np.isfinite(variance_flat)

                combined_mask = mask_finite & mask_finite_var
                if np.sum(combined_mask) < 5:
                    log_likelihood_mapp[i, v] = np.nan
                    continue

                if matrix_components == 1:
                    fit_model_mask = np.atleast_2d(model_matrix[v][i][combined_mask]).T  # shape (nPix_mask, 1)
                elif matrix_components == 2:
                    # accomodate shapes; I assume model_matrix[v] returns an array with shape (2, nObs, nPix)
                    m0 = model_matrix[v][0, i, combined_mask]
                    m1 = model_matrix[v][1, i, combined_mask]
                    fit_model_mask = np.vstack((m0, m1)).T  # shape (nPix_mask, 2)
                else:
                    print("matrix_components must be 1 or 2")
                    log_likelihood_mapp[i, v] = np.nan
                    continue

                output = self.likelihood_calculator(d=observation_matrix[i][combined_mask],
                                                    M=fit_model_mask,
                                                    Sigma_0_flat=variance_flat[combined_mask],
                                                    log_sigma_0=log_sigma,
                                                    prior_psi=prior, gamma=2)
                log_likelihood_mapp[i, v] = output

        return log_likelihood_mapp


##### 0.1.2 Functions following classic extraction strategy

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def normalize_profile(profile):
    total = np.sum(np.nan_to_num(profile, nan=0.0))
    if total <=0:
        print ('WARNING: The sum of this curve is not positive')
        '''
        mask = (profile>0)
        total_posi = np.sum(np.nan_to_num(profile[mask], nan=0.0))

        PSF = np.zeros_like(profile)

        PSF[mask] = profile[mask]/ total_posi

        return PSF
        '''
        return profile / total
    else:
        return profile / total

def moffat(x, A, alpha, beta, B, x0):
    return A * (1.0 + ((x - x0) / alpha)**2)**(-beta) + B

def fit_integrated(spec2d, x0_known=None,
                          beta_init=3.0, fwhm_guess=3.0, fit_x0=False, fit_x0_planet=True, wave=None,
                          mask_planet_window=None, fit_planet_component=False, fit_star_component = True, 
                          x0_known_planet=None, fit_method_star='moffat', fit_method_planet='moffat'):
    """
    Fit integrated profile. Returns (params_dict, profile, model_norm_full, model_planet_norm, model_background_poly)
    - fit_method_star: 'moffat' or 'polynomial' for the stellar fit
    - fit_method_planet: 'moffat' or 'polynomial' for the planet residual fit
    """
    n_spatial, n_wave = spec2d.shape
    x = np.arange(n_spatial)

    spec2d = np.nan_to_num(spec2d, nan=0.0)
    profile = np.trapz(spec2d, x=wave, axis=1)

    B0 = 0.0
    A0 = max(np.max(profile) - B0, 1.0)
    alpha0 = fwhm_guess / (2.0 * np.sqrt(2.0**(1.0 / beta_init) - 1.0))

    # Mask the planet region if provided
    if mask_planet_window is not None:
        mask_min, mask_max = mask_planet_window
        mask = (x < mask_min) | (x > mask_max)
        x_mask = x[mask]
        profile_mask = profile[mask]
    else:
        mask = np.zeros_like(x, dtype=bool)
        x_mask = x.copy()
        profile_mask = profile.copy()

    # initial x0 (for star)
    if x0_known is None:
        # argmax in masked region -> return x coordinate
        x0_known = int(x_mask[np.nanargmax(profile_mask)])

    # set defaults for many outputs so they're always defined
    A = alpha = beta = B = x0f = fwhm = None
    popt = pcov = None
    model_norm_full = np.zeros_like(x)
    model_full = np.zeros_like(x)
    model_background_poly = np.zeros_like(x)

    # --- Stellar fit ---
    if fit_star_component == True:
        if fit_x0:
            if fit_method_star.lower() == 'moffat':
                # bounds for x0 must be in coordinate space, not indices of profile_mask
                x0_low = max(0, x0_known - 5)
                x0_high = min(x_mask.max(), x0_known + 5)
                p0 = [A0, alpha0, beta_init, x0_known]
                lower = [0.0, 1e-6, 0.5, x0_low]
                upper = [np.nanmax(profile_mask)*10.0, np.inf, 50.0, x0_high]
                popt, pcov = curve_fit(lambda xx, A_, alpha_, beta_, x0_:
                                    moffat(xx, A_, alpha_, beta_, B0, x0_),
                                    x_mask, profile_mask, p0=p0, bounds=(lower, upper), maxfev=20000)
                A, alpha, beta, x0f = popt
            elif fit_method_star.lower() == 'polynomial':
                # polynomial main fit (deg 5) - use Polynomial.fit and convert to power basis
                deg = 5
                if x_mask.size == 0:
                    coeffs = np.zeros(deg+1)
                else:
                    poly_fit = Polynomial.fit(x_mask, profile_mask, deg=deg)
                    poly_unscaled = poly_fit.convert(domain=[np.min(x_mask), np.max(x_mask)])
                    coeffs = poly_unscaled.coef
                    coeffs = np.pad(coeffs, (0, max(0, deg+1 - coeffs.size)), constant_values=0.0)[:deg+1]
                model_full = Polynomial(coeffs)(x)
                model_norm_full = normalize_profile(model_full)
                A = alpha = beta = 0.0
                B = coeffs[0] if coeffs.size>0 else 0.0
                x0f = x0_known
                popt = coeffs
                pcov = None
                model_background_poly = np.zeros_like(x)
        else:
            # x0 fixed at x0_known
            if fit_method_star.lower() == 'moffat':
                p0 = [A0, alpha0, beta_init, B0]
                lower = [0.0, 1e-6, 0.5, -np.inf]
                upper = [np.inf, np.inf, 50.0, np.inf]
                popt, pcov = curve_fit(lambda xx, A_, alpha_, beta_, B_:
                                    moffat(xx, A_, alpha_, beta_, B_, x0_known),
                                    x_mask, profile_mask, p0=p0, bounds=(lower, upper), maxfev=20000)
                A, alpha, beta, B = popt
                x0f = x0_known
                model_normal_full = moffat(x, A, alpha, beta, B0, x0f)
                model_norm_full = normalize_profile(model_normal_full)
            elif fit_method_star.lower() == 'polynomial':
                deg = 5
                if x_mask.size == 0:
                    coeffs = np.zeros(deg+1)
                else:
                    poly_fit = Polynomial.fit(x_mask, profile_mask, deg=deg)
                    poly_unscaled = poly_fit.convert(domain=[np.min(x_mask), np.max(x_mask)])
                    coeffs = poly_unscaled.coef
                    coeffs = np.pad(coeffs, (0, max(0, deg+1 - coeffs.size)), constant_values=0.0)[:deg+1]
                model_full = Polynomial(coeffs)(x)
                model_norm_full = normalize_profile(model_full)
                A = alpha = beta = 0.0
                B = coeffs[0] if coeffs.size>0 else 0.0
                x0f = x0_known
                popt = coeffs
                pcov = None
                model_background_poly = np.zeros_like(x)
        # If stellar fit was via fit_x0+moffat, compute model_norm_full now (that branch didn't set it above)
        if fit_method_star.lower() == 'moffat' and fit_x0:
            model_normal_full = moffat(x, A, alpha, beta, B0, x0f)
            model_norm_full = normalize_profile(model_normal_full)

        # Ensure model_norm_full defined
        if model_norm_full is None or np.all(model_norm_full == 0):
            # fallback: normalize profile itself
            model_norm_full = normalize_profile(model_full) if np.any(model_full) else normalize_profile(profile*0.0)
    else:
        model_norm_full = 0.0
        A = alpha = beta = B0 = x0f = fwhm = popt = pcov = None


    # --- Planet fit (residual) ---
    if fit_planet_component:
        Residual = normalize_profile(profile) - model_norm_full

        x_mask_planet = x[~mask]  # unmasked region (where planet should be)
        profile_mask_planet = Residual[~mask]

        # initial planet guess
        if x0_known_planet is None:
            if profile_mask_planet.size == 0:
                x0_known_planet = 0
            else:
                local_idx = np.nanargmax(profile_mask_planet)
                x0_known_planet = int(x_mask_planet[local_idx])
        print('The initial guess of the planet position is pixel:', x0_known_planet)

        # amplitude and alpha initial guesses for planet
        A_upper_bound = max(Residual[x0_known_planet],0.0004)
        B0_p = 0.0
        A0_p = max((Residual[x0_known_planet]) - B0_p, 1e-12)
        alpha0_p = fwhm_guess / (2.0 * np.sqrt(2.0**(1.0 / beta_init) - 1.0))

        # Ensure x_mask_planet non-empty
        if x_mask_planet.size == 0:
            # nothing to fit; set zeros
            A_p = alpha_p = beta_p = 0.0
            x0f_p = x0_known_planet
            fwhm_p = 0.0
            popt_p = pcov_p = None
            model_planet_norm = np.zeros_like(x)
            model_background_poly = np.zeros_like(x)
        else:
            # determine safe x0 bounds in coordinate space from x_mask_planet
            x_min_fit = np.min(x_mask_planet)
            x_max_fit = np.max(x_mask_planet)
            # clamp x0 initial guess to that domain
            x0_known_planet = int(np.clip(x0_known_planet, x_min_fit, x_max_fit))

            if str(fit_method_planet).lower() == 'moffat':
                # pure moffat fit to residual
                if fit_x0_planet:
                    x0_low = max(x_min_fit, x0_known_planet - 5)
                    x0_high = min(x_max_fit, x0_known_planet + 5)
                    p0 = [A0_p, alpha0_p, beta_init, x0_known_planet]
                    lower = [0.0, 1e-8, 0.5, x0_low]
                    upper = [A_upper_bound, np.inf, 50.0, x0_high]
                    popt_p, pcov_p = curve_fit(lambda xx, A_, alpha_, beta_, x0_:
                                               moffat(xx, A_, alpha_, beta_, B0_p, x0_),
                                               x_mask_planet, profile_mask_planet, p0=p0, bounds=(lower, upper), maxfev=20000)
                    A_p, alpha_p, beta_p, x0f_p = popt_p
                else:
                    p0 = [A0_p, alpha0_p, beta_init]
                    lower = [0.0, 1e-8, 0.5]
                    upper = [np.nanmax(profile_mask_planet)*10.0, np.inf, 50.0]
                    popt_p, pcov_p = curve_fit(lambda xx, A_, alpha_, beta_:
                                               moffat(xx, A_, alpha_, beta_, B0_p, x0_known_planet),
                                               x_mask_planet, profile_mask_planet, p0=p0, bounds=(lower, upper), maxfev=20000)
                    A_p, alpha_p, beta_p = popt_p
                    x0f_p = x0_known_planet

                fwhm_p = 2.0 * alpha_p * np.sqrt(2.0**(1.0/beta_p) - 1.0)
                model_planet_norm = moffat(x, A_p, alpha_p, beta_p, B0_p, x0f_p)
                model_background_poly = Residual - model_planet_norm

            elif str(fit_method_planet).lower() == 'moffat+polynomial':
                # Moffat(planet) + 3rd-order polynomial background
                # We'll fit parameters: A, alpha, beta, (optionally) x0, c0, c1, c2, c3
                x_ptp_planet = np.ptp(x_mask_planet)
                if x_ptp_planet == 0:
                    x_ptp_planet = 1.0

                if fit_x0_planet:
                    # Fit x0 as parameter, but bounds must be within [x_min_fit, x_max_fit]
                    x0_low = max(x_min_fit, x0_known_planet - 5)
                    x0_high = min(x_max_fit, x0_known_planet + 5)
                    def model_func(xx, A_, alpha_, beta_, x0_, c0_, c1_, c2_, c3_, c4_, c5_, c6_, c7_):
                        poly = c0_ + c1_*xx + c2_*xx**2 + c3_*xx**3 + c4_*xx**4 + c5_*xx**5 + c6_*xx**6 + c7_*xx**7
                        poly[x0_known_planet-10:x0_known_planet+10] += moffat(xx[x0_known_planet-10:x0_known_planet+10], A_, alpha_, beta_, 0.0, x0_)
                        return poly
                    

                    print (x_mask_planet, x0_known_planet)
                    
                    p0 = [A0_p, alpha0_p, beta_init, x0_known_planet, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
                    lower = [0.0, 1e-8, 0.5, x0_low, min(Residual[0], 0.0), -10*A0_p/(x_ptp_planet), -10*A0_p/(x_ptp_planet**2), -10*A0_p/(x_ptp_planet**3), -10*A0_p/(x_ptp_planet**4), -10*A0_p/(x_ptp_planet**5), -10*A0_p/(x_ptp_planet**6), -10*A0_p/(x_ptp_planet**7)]
                    upper = [A_upper_bound, np.inf, 50.0, x0_high, max(Residual[0], 0.0002), 10*A0_p/x_ptp_planet, 10*A0_p/(x_ptp_planet**2), 10*A0_p/(x_ptp_planet**3), 10*A0_p/(x_ptp_planet**4), 10*A0_p/(x_ptp_planet**5), 10*A0_p/(x_ptp_planet**6), 10*A0_p/(x_ptp_planet**7)]
                    bounds = (np.array(lower, dtype=float), np.array(upper, dtype=float))
                    popt_p, pcov_p = curve_fit(model_func, x_mask_planet, profile_mask_planet, p0=p0, bounds=bounds, maxfev=200000)
                    A_p, alpha_p, beta_p, x0f_p, c0_p, c1_p, c2_p, c3_p, c4_p, c5_p, c6_p, c7_p = popt_p

                else:
                    # x0 fixed at x0_known_planet
                    def model_func(xx, A_, alpha_, beta_, c0_, c1_, c2_, c3_, c4_, c5_, c6_, c7_):
                        poly = c0_ + c1_*xx + c2_*xx**2 + c3_*xx**3 + c4_*xx**4 + c5_*xx**5 + c6_*xx**6 + c7_*xx**7
                        poly[x0_known_planet-10:x0_known_planet+10] += moffat(xx[x0_known_planet-10:x0_known_planet+10], A_, alpha_, beta_, 0.0, x0_)
                        return poly
                    p0 = [A0_p, alpha0_p, beta_init, 0., 0., 0., 0., 0., 0., 0.]
                    lower = [0.0, 1e-8, 0.5, -10*A0_p, -10*A0_p/x_ptp_planet, -10*A0_p/(x_ptp_planet**2), -10*A0_p/(x_ptp_planet**3), -10*A0_p/(x_ptp_planet**4), -10*A0_p/(x_ptp_planet**5), -10*A0_p/(x_ptp_planet**6), -10*A0_p/(x_ptp_planet**7)]
                    upper = [A_upper_bound, np.inf, 50.0, 10*A0_p, 10*A0_p/x_ptp_planet, 10*A0_p/(x_ptp_planet**2), 10*A0_p/(x_ptp_planet**3), 10*A0_p/(x_ptp_planet**4), 10*A0_p/(x_ptp_planet**5), 10*A0_p/(x_ptp_planet**6), 10*A0_p/(x_ptp_planet**7)]
                    bounds = (np.array(lower, dtype=float), np.array(upper, dtype=float))
                    popt_p, pcov_p = curve_fit(model_func, x_mask_planet, profile_mask_planet, p0=p0, bounds=bounds, maxfev=200000)
                    A_p, alpha_p, beta_p, x0f_p, c0_p, c1_p, c2_p, c3_p, c4_p, c5_p, c6_p, c7_p = popt_p
                    x0f_p = x0_known_planet
            

                # compute outputs

                fwhm_p = 2.0 * alpha_p * np.sqrt(2.0**(1.0/beta_p) - 1.0)
                if fit_x0_planet:
                    poly_vals = c0_p + c1_p*x + c2_p*x**2 + c3_p*x**3 + c4_p*x**4 + c5_p*x**5 + c6_p*x**6 + c7_p*x**7
                    model_planet_norm = moffat(x, A_p, alpha_p, beta_p, 0.0, x0f_p)
                    model_background_poly = poly_vals
                else:
                    poly_vals = c0_p + c1_p*x + c2_p*x**2 + c3_p*x**3
                    model_planet_norm = moffat(x, A_p, alpha_p, beta_p, 0.0, x0f_p)
                    model_background_poly = poly_vals

            elif str(fit_method_planet).lower() == 'double polynomial':
                # Two polynomial fit kernel tto fit both the planet and background
                x_ptp_planet = np.ptp(x_mask_planet)
                if x_ptp_planet == 0:
                    x_ptp_planet = 1.0

                deg = 7
                if fit_x0_planet:
                    print("WARNING: Cannot fit x0 with polynomial fitting. Fitting polynomial (degree 7) instead.")
                # Fit degree-7 polynomial to the masked data using numpy.polynomial
                def model_func(xx, c0_, c1_, c2_, c3_, c4_, c5_, c6_, c7_):
                    x_planet_core = xx[x0_known_planet-10:x0_known_planet+10] 
                    poly_planet = c0_ + c1_*x_planet_core + c2_*x_planet_core**2 + c3_*x_planet_core**3 + c4_*x_planet_core**4 + c5_*x_planet_core**5 + c6_*x_planet_core**6 + c7_*x_planet_core**7
                    poly_background = c0_ + c1_*xx + c2_*xx**2 + c3_*xx**3 + c4_*xx**4 + c5_*xx**5 

                    poly_background [x0_known_planet-10:x0_known_planet+10] += poly_planet

                    return  poly_background
                
                p0 = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
                lower = [-10*A0_p, -10*A0_p/x_ptp_planet, -10*A0_p/(x_ptp_planet**2), -10*A0_p/(x_ptp_planet**3), -10*A0_p/(x_ptp_planet**4), -10*A0_p/(x_ptp_planet**5), -10*A0_p/(x_ptp_planet**6), -10*A0_p/(x_ptp_planet**7)]
                upper = [10*A0_p, 10*A0_p/x_ptp_planet, 10*A0_p/(x_ptp_planet**2), 10*A0_p/(x_ptp_planet**3), 10*A0_p/(x_ptp_planet**4), 10*A0_p/(x_ptp_planet**5), 10*A0_p/(x_ptp_planet**6), 10*A0_p/(x_ptp_planet**7)]
                bounds = (np.array(lower, dtype=float), np.array(upper, dtype=float))
                popt_p, pcov_p = curve_fit(model_func, x_mask_planet, profile_mask_planet, p0=p0, bounds=bounds, maxfev=200000)
                c0_p, c1_p, c2_p, c3_p, c4_p, c5_p, c6_p, c7_p = popt_p
                A_p = alpha_p = beta_p = x0f_p = 0.0  

                fwhm_p = 0.0
                poly_planet = c0_p + c1_p*x + c2_p*x**2 + c3_p*x**3 + c4_p*x**4 + c5_p*x**5 + c6_p*x**6 + c7_p*x**7
                poly_background = c0_p + c1_p*x + c2_p*x**2 + c3_p*x**3 + c4_p*x**4 + c5_p*x**5
                model_background_poly = poly_background          
                model_planet_norm = poly_planet
            
            else:
                # unknown planet method
                print(f"Unknown fit_method_planet='{fit_method_planet}'. No planet fit performed.")
                A_p = alpha_p = beta_p = B0_p = x0f_p = fwhm_p = 0.0
                popt_p = pcov_p = None
                model_planet_norm = np.zeros_like(x)
                model_background_poly = np.zeros_like(x)
    else:
        # no planet component
        A_p = alpha_p = beta_p = B0_p = x0f_p = fwhm_p = popt_p = pcov_p = None
        model_planet_norm = np.zeros_like(x)
        model_background_poly = np.zeros_like(x)

    # final return (similar structure to yours)
    return {
        'A': [A, A_p], 'alpha': [alpha, alpha_p], 'beta': [beta, beta_p],
        'B': [B0, B0_p if 'B0_p' in locals() else 0.0],
        'x0': [x0f, x0f_p if 'x0f_p' in locals() else None],
        'fwhm': [fwhm, fwhm_p if 'fwhm_p' in locals() else None],
        'popt': [popt, popt_p if 'popt_p' in locals() else None],
        'pcov': [pcov, pcov_p if 'pcov_p' in locals() else None],
        'coeff': [c0_p, c1_p, c2_p, c3_p, c4_p, c5_p, c6_p, c7_p]
    }, profile, model_norm_full, model_planet_norm, model_background_poly





### 1. Extract the spectrum using 0.1.2 functions (Classic Extraction)

#### 1.1 Mask the low transmission wavelengths

In [ ]:
#--------------------
#load the wavelength data and the transmission templates
night='2023-01-01'
workpath =f'/data2/peng/{night}'

prep_star = Spectra_Preparator (folder_name='2023-01-01', T=3700, G=4.0, Z=-0.0, path='/data2/peng')

wave_cal = fits.open(workpath + '/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
wave_data=wave_cal[1].data

#trans = fits.open(workpath + '/cal/TRANSM_SPEC.fits')
trans = fits.open(workpath +f'/chi_tau/{night}/out/molecfit/TELLURIC_DATA.fits')

trans_data = trans[1].data

plt.plot(trans_data['lambda'], trans_data['mtrans'], linewidth=0.5)
plt.show()

band=-3
wave_range=info.lambda_range['wave'][band]
order_range=info.lambda_range['order'][band]

#Prepare the transmission spectra into the shape of observation matrix

trans_matrix = prep_star.spectrum_cutter(wave=trans_data['lambda']*1e3, flux=trans_data['mtrans'], saving_file=True, lam_range=wave_range,
filenames='/Trans.npy', ignore_detector=False, Wave_solution=wave_data,
saving_path=prep_star.path+'/spectra_input')



fit_workflow_A = Spectra_fitting(spectra2D=flux2d_A, transmission_matrix=trans_matrix, window_length=300, polyorder=2)
spectra_masked_A=fit_workflow_A.Mask_spec(atm_threshold=0.7,  sigma_threshold=3, Mask_bad_pxiel=True)



fig, axes = plt.subplots(7,3, figsize=(12,18))

plt.subplots_adjust(hspace=0.3)
plt.suptitle(f'Positon A, Night {night}', fontsize=10)
for i in range(3):
    for j in range(7):
        ax=axes[j,i]
        wl=wave_data[i,j]
        img=spectra_masked_A[i][j]
        map=ax.imshow(img, aspect='auto',norm=SymLogNorm(5, 500), extent=[wl.min(), wl.max(), 0, img.shape[0]])

        ax.set_title('Order %s, Detector %s'%((23+j), (i+1)), fontsize=6) 

        # X axis: smaller font, ~3–4 ticks, integer values
        ax.tick_params(axis="x", labelsize=8)
        ax.locator_params(axis="x", nbins=4, integer=True)
        
        # Label only outer axes to reduce clutter
        if j == 6:  # bottom row
            ax.set_xlabel("Wavelength", fontsize=9)

        if i == 0:  # first column
            ax.set_ylabel("Spatial position", fontsize=9)


        plt.colorbar(map)

#plt.tight_layout()       
plt.show()

#-----------------------

flux2d_B_pad = np.zeros(shape=flux2d_A.shape)

for i in range(flux2d_B_pad.shape[0]):

    for j in range (flux2d_B_pad.shape[1]):

        arr=flux2d_B[i][j].shape[0]

        flux2d_B_pad[i][j][0:arr] = flux2d_B[i][j]

fit_workflow_B = Spectra_fitting(spectra2D=flux2d_B_pad, transmission_matrix=trans_matrix, window_length=300, polyorder=2)
spectra_masked_B=fit_workflow_B.Mask_spec(atm_threshold=0.6,  sigma_threshold=1, Mask_bad_pxiel=True)

fig, axes = plt.subplots(7,3, figsize=(12,18))

plt.subplots_adjust(hspace=0.3)
plt.suptitle(f'Positon B, Night {night}', fontsize=10)
for i in range(3):
    for j in range(7):
        ax=axes[j,i]
        wl=wave_data[i,j]
        img=spectra_masked_B[i][j]
        map=ax.imshow(img, aspect='auto',norm=SymLogNorm(5, 500), extent=[wl.min(), wl.max(), 0, img.shape[0]])

        ax.set_title('Order %s, Detector %s'%((23+j), (i+1)), fontsize=6) 


        # X axis: smaller font, ~3–4 ticks, integer values
        ax.tick_params(axis="x", labelsize=8)
        ax.locator_params(axis="x", nbins=4, integer=True)
        
        # Label only outer axes to reduce clutter
        if j == 6:  # bottom row
            ax.set_xlabel("Wavelength", fontsize=9)
    
        if i == 0:  # first column
            ax.set_ylabel("Spatial position", fontsize=9)

        plt.colorbar(map)

#plt.tight_layout()       
plt.show()


#### 1.2 Generate stellar master templates and drop off order 29, 28 

In [ ]:
master_matrix_A = fit_workflow_A.stellar_master_spec(wave=wave_data, cut_off=2)

master_matrix_B = fit_workflow_B.stellar_master_spec(wave=wave_data, cut_off=2)

#combine_A_B= fit_workflow_A.spectra[:,0:5]+fit_workflow_B.spectra[:,0:5]

fit_workflow_A.spectra=fit_workflow_A.spectra[:,0:5]
fit_workflow_B.spectra=fit_workflow_B.spectra[:,0:5]

#### 1.3 Fit and Subtract the planet PSF

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt


def normalize_profile(profile):
    """Normalize a 1D profile so its absolute-value sum equals 1."""
    total = float(np.nansum(np.abs(np.nan_to_num(profile, nan=0.0))))
    return profile / total if total > 1e-30 else profile


def moffat(x, A, alpha, beta, B, x0):
    return A * (1.0 + ((x - x0) / alpha)**2)**(-beta) + B


def fit_stellar_halo_and_subtract(spec2d, x0_planet=19, fwhm_guess=8.0, beta_init=3.0,
                                   sky_rows=None, poly_deg=3):
    """
    Method 5.A.2 — Two-step stellar halo removal + physical-unit OLS subtraction.

    Step 1 : Unconstrained polyfit (degree poly_deg) to sky rows → halo shape.
             Uses numpy's stable least-squares solver; no bounds needed → never diverges.
    Step 2 : Moffat fit to the residual (data − polynomial) near x0_planet
             → planet center x0 and FWHM for the extraction aperture.
    Step 3 : OLS per-column halo subtraction on physical spec2d [e-/s] using
             the polynomial shape → clean_phys in [e-/s] (Bug 1 fix).

    Profile used for fitting and display: mean over wavelengths [e-/s].
    Values are O(10) — consistent with the raw flux counts in "0. Load the data".

    Parameters
    ----------
    spec2d    : (n_spatial, n_wave)  raw 2D spectrum in physical units [e-/s]
    x0_planet : int    initial guess for companion spatial pixel
    fwhm_guess: float  initial FWHM guess [pixels]
    beta_init : float  initial Moffat beta
    sky_rows  : list   spatial rows free of companion signal (used for polyfit + OLS)
    poly_deg  : int    polynomial degree for halo model (3 is stable for ~10 sky rows)

    Returns
    -------
    clean_phys   : (n_spatial, n_wave)  halo-subtracted data in physical units [e-/s]
    poly_shape   : (n_spatial,)         unit-L1 polynomial halo shape used for OLS
    moffat_shape : (n_spatial,)         unit-L1 Moffat planet shape (for reference)
    fit_params   : dict  keys: 'x0', 'fwhm', 'alpha', 'beta', 'A_norm',
                              'poly_c_norm', 'profile_total'
    profile_mean : (n_spatial,)         mean-over-wavelength profile [e-/s], values O(10)
    halo_amps    : (n_wave,)            fitted halo amplitude per column [e-/s]
    """
    n_spatial, n_wave = spec2d.shape
    x      = np.arange(n_spatial, dtype=float)
    x_ptp  = max(float(np.ptp(x)), 1.0)
    x_norm = x / x_ptp    # [0, 1] for stable poly coefficients

    if sky_rows is None:
        sky_rows = list(range(0, 5)) + list(range(25, 30))
    sky_mask = np.zeros(n_spatial, dtype=bool)
    for r in sky_rows:
        if 0 <= r < n_spatial:
            sky_mask[r] = True

    spec_arr = np.nan_to_num(spec2d, nan=0.0)

    # ── Mean profile in physical units [e-/s]  (values O(10) = raw flux counts) ──
    profile_mean  = np.mean(spec_arr, axis=1)
    profile_total = max(float(np.nansum(np.abs(profile_mean))), 1e-30)
    profile_norm  = profile_mean / profile_total   # dimensionless, for stable fitting

    # ── Step 1: Stable polyfit to sky rows ────────────────────────────────────
    # numpy's lstsq-based polyfit always converges and never needs bounds.
    # With poly_deg=3 and ~10 sky rows the system is well-conditioned.
    poly_c        = np.polyfit(x_norm[sky_mask], profile_norm[sky_mask], deg=poly_deg)
    poly_norm_fit = np.polyval(poly_c, x_norm)   # normalised halo model

    poly_total = max(float(np.nansum(np.abs(poly_norm_fit))), 1e-30)
    poly_shape = poly_norm_fit / poly_total       # unit-L1 shape for OLS

    # ── Step 2: Moffat fit to residual at the planet window ──────────────────
    residual = profile_norm - poly_norm_fit
    lo_p  = max(0,          x0_planet - 10)
    hi_p  = min(n_spatial,  x0_planet + 10)
    x_win   = x[lo_p:hi_p]
    res_win = residual[lo_p:hi_p]

    # Initial x0 from peak of residual; then refine with Moffat fit
    if res_win.size > 0 and np.nanmax(res_win) > 0:
        x0f_p = float(x_win[np.nanargmax(res_win)])
    else:
        x0f_p = float(x0_planet)

    alpha0  = fwhm_guess / (2.0 * np.sqrt(2.0**(1.0 / beta_init) - 1.0))
    A0      = max(float(residual[int(np.clip(x0f_p, 0, n_spatial - 1))]), 1e-12)
    A_upper = max(A0 * 10.0, float(np.nanmax(np.abs(profile_norm))), 1e-4)

    try:
        popt_m, _ = curve_fit(
            lambda xx, A_, a_, b_, x0_: A_ * (1.0 + ((xx - x0_) / a_)**2)**(-b_),
            x_win, res_win,
            p0=[A0, alpha0, beta_init, x0f_p],
            bounds=([0., 1e-6, 0.5, float(lo_p)],
                    [A_upper, np.inf, 50., float(hi_p - 1)]),
            maxfev=20000)
        A_p, alpha_p, beta_p, x0f_p = popt_m
        fwhm = 2.0 * alpha_p * np.sqrt(2.0**(1.0 / beta_p) - 1.0)
    except (RuntimeError, ValueError) as exc:
        print(f"    WARNING: Moffat fit failed ({exc}); using initial guess.")
        A_p, alpha_p, beta_p = A0, alpha0, beta_init
        fwhm = fwhm_guess

    moff_norm_fit = A_p * (1.0 + ((x - x0f_p) / alpha_p)**2)**(-beta_p)
    moff_total    = max(float(np.nansum(np.abs(moff_norm_fit))), 1e-30)
    moffat_shape  = moff_norm_fit / moff_total

    fit_params = {
        'x0'           : [None, x0f_p],
        'fwhm'         : [None, fwhm],
        'alpha'        : [None, alpha_p],
        'beta'         : [None, beta_p],
        'A_norm'       : [None, A_p],
        'poly_c_norm'  : poly_c,
        'profile_total': profile_total,
    }

    # ── Step 3: OLS per-column halo subtraction on PHYSICAL 2D data ──────────
    # Fit scalar amplitude of poly_shape to sky rows of each wavelength column,
    # then subtract: clean_phys = spec_arr − poly_shape * amp(λ).
    # clean_phys is in [e-/s] — physical units required by optimal_extraction
    # so that its internal Poisson term (|D|/gain) is correct (Bug 1 fix).
    '''
    P_sky     = poly_shape[sky_mask]                         # (n_sky,)
    PdotP     = float(np.dot(P_sky, P_sky))
    d_sky     = spec_arr[sky_mask, :]                        # (n_sky, n_wave)
    halo_amps = (np.dot(P_sky, d_sky) / PdotP
                 if PdotP > 1e-30 else np.zeros(n_wave))    # (n_wave,) [e-/s]

    clean_phys = spec_arr - np.outer(poly_shape, halo_amps)
    clean_phys[~np.isfinite(spec2d)] = np.nan

    return clean_phys, poly_shape, moffat_shape, fit_params, profile_mean, halo_amps
    '''

     # ── Step 3: Per-column polyfit halo subtraction on PHYSICAL 2D data ────────
    # Fit the polynomial independently for each wavelength column using the sky
    # rows. This captures wavelength-dependent changes in halo shape (e.g. from
    # PSF width variation or differential atmospheric dispersion) that the
    # previous fixed-shape OLS approach could not handle.
    # clean_phys is in [e-/s] — physical units required by optimal_extraction
    # so that its internal Poisson term (|D|/gain) is correct (Bug 1 fix).
    V_sky = np.vander(x_norm[sky_mask], poly_deg + 1)        # (n_sky, deg+1)
    V_all = np.vander(x_norm,           poly_deg + 1)        # (n_spa, deg+1)
    C_cols, _, _, _ = np.linalg.lstsq(V_sky, spec_arr[sky_mask, :], rcond=None)
                                                              # (deg+1, n_wave)
    halo_model = V_all @ C_cols                              # (n_spa, n_wave) [e-/s]

    # halo_amps: OLS amplitudes retained for return-value compatibility
    P_sky     = poly_shape[sky_mask]                         # (n_sky,)
    PdotP     = float(np.dot(P_sky, P_sky))
    d_sky     = spec_arr[sky_mask, :]                        # (n_sky, n_wave)
    halo_amps = (np.dot(P_sky, d_sky) / PdotP
                 if PdotP > 1e-30 else np.zeros(n_wave))    # (n_wave,) [e-/s]

    clean_phys = spec_arr - halo_model
    clean_phys[~np.isfinite(spec2d)] = np.nan

    return clean_phys, poly_shape, moffat_shape, fit_params, profile_mean, halo_amps

# ─────────────────────────────────────────────────────────────────────────────
# Position A  (2022-12-31, nod A)
# ─────────────────────────────────────────────────────────────────────────────
nDet   = fit_workflow_A.spectra.shape[0]
nOrd   = fit_workflow_A.spectra.shape[1]
n_spa  = 35           # spatial rows extracted; shared with cell 26
n_wave = fit_workflow_A.spectra.shape[3]

spectra_clean_A_phys = np.zeros((nDet, nOrd, n_spa, n_wave))
moffat_params_A      = np.empty((nDet, nOrd), dtype=object)
poly_shapes_A        = np.empty((nDet, nOrd), dtype=object)

sky_rows_A = list(range(0, 10)) + list(range(27, 35))   # rows free of companion signal

fig, axes = plt.subplots(nDet, nOrd, figsize=(20, 10))

for iDet in range(nDet):
    for iOrder in range(nOrd):
        spec2d = fit_workflow_A.spectra[iDet, iOrder, 0:n_spa, :]

        (clean_phys, poly_shape, moffat_shape,
         fit_params, profile_mean, halo_amps) = fit_stellar_halo_and_subtract(
            spec2d, x0_planet=19, fwhm_guess=8.0,
            sky_rows=sky_rows_A, poly_deg=3)

        spectra_clean_A_phys[iDet, iOrder] = clean_phys
        moffat_params_A[iDet, iOrder]      = fit_params
        poly_shapes_A[iDet, iOrder]        = poly_shape

        # ── Diagnostic in physical [e-/s] — same scale as "0. Load the data" ─
        ptot    = fit_params['profile_total']   # = sum(|mean_profile|)
        x_arr   = np.arange(n_spa, dtype=float)
        x_nrm   = x_arr / max(float(np.ptp(x_arr)), 1.0)
        poly_rec = np.polyval(fit_params['poly_c_norm'], x_nrm) * ptot   # [e-/s]
        moff_rec = (fit_params['A_norm'][1]
                    * (1.0 + ((x_arr - fit_params['x0'][1])
                               / fit_params['alpha'][1])**2)
                    ** (-fit_params['beta'][1])) * ptot                    # [e-/s]

        ax = axes[iDet, iOrder]
        ax.plot(profile_mean,                    color='blue',   lw=1.2, alpha=0.7, label='Mean profile')
        ax.plot(poly_rec,                        color='red',    lw=1.0, ls='--',   label='Poly halo')
        ax.plot(poly_rec + moff_rec,             color='orange', lw=1.0,            label='Poly+Moffat')
        ax.plot(np.mean(clean_phys, axis=1),     color='green',  lw=1.0, ls=':',    label='Cleaned mean')
        ax.axhline(0, color='lightgray', ls='--', lw=0.5)
        ax.set_xlim(0, n_spa - 1)
        ax.set_title(f'Det{iDet+1} Ord{iOrder+1}  FWHM={fit_params["fwhm"][1]:.1f}px', fontsize=7)
        ax.set_xlabel('Spatial px', fontsize=6)
        ax.set_ylabel('Mean flux [e-/s]', fontsize=6)
        ax.tick_params(labelsize=5)
        ax.legend(fontsize=4)

plt.suptitle('Position A - 2023-01-01 - 5.A.2: polyfit halo + Moffat residual, physical OLS', fontsize=9)
plt.subplots_adjust(hspace=0.55, wspace=0.45)
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# Position B  (2022-12-31, nod B)
# ─────────────────────────────────────────────────────────────────────────────
nDet_B   = fit_workflow_B.spectra.shape[0]
nOrd_B   = fit_workflow_B.spectra.shape[1]
n_wave_B = fit_workflow_B.spectra.shape[3]

spectra_clean_B_phys = np.zeros((nDet_B, nOrd_B, n_spa, n_wave_B))
moffat_params_B      = np.empty((nDet_B, nOrd_B), dtype=object)
poly_shapes_B        = np.empty((nDet_B, nOrd_B), dtype=object)

sky_rows_B = list(range(0, 10)) + list(range(27, 35))   # nod-B rows free of companion signal

fig, axes = plt.subplots(nDet_B, nOrd_B, figsize=(20, 10))

for iDet in range(nDet_B):
    for iOrder in range(nOrd_B):
        spec2d = fit_workflow_B.spectra[iDet, iOrder, 0:n_spa, :]

        (clean_phys, poly_shape, moffat_shape,
         fit_params, profile_mean, halo_amps) = fit_stellar_halo_and_subtract(
            spec2d, x0_planet=19, fwhm_guess=8.0,
            sky_rows=sky_rows_B, poly_deg=3)

        spectra_clean_B_phys[iDet, iOrder] = clean_phys
        moffat_params_B[iDet, iOrder]      = fit_params
        poly_shapes_B[iDet, iOrder]        = poly_shape

        # ── Diagnostic ─────────────────────────────────────────────────────────
        ptot    = fit_params['profile_total']
        x_arr   = np.arange(n_spa, dtype=float)
        x_nrm   = x_arr / max(float(np.ptp(x_arr)), 1.0)
        poly_rec = np.polyval(fit_params['poly_c_norm'], x_nrm) * ptot
        moff_rec = (fit_params['A_norm'][1]
                    * (1.0 + ((x_arr - fit_params['x0'][1])
                               / fit_params['alpha'][1])**2)
                    ** (-fit_params['beta'][1])) * ptot

        ax = axes[iDet, iOrder]
        ax.plot(profile_mean,                    color='blue',   lw=1.2, alpha=0.7, label='Mean profile')
        ax.plot(poly_rec,                        color='red',    lw=1.0, ls='--',   label='Poly halo')
        ax.plot(poly_rec + moff_rec,             color='orange', lw=1.0,            label='Poly+Moffat')
        ax.plot(np.mean(clean_phys, axis=1),     color='green',  lw=1.0, ls=':',    label='Cleaned mean')
        ax.axhline(0, color='lightgray', ls='--', lw=0.5)
        ax.set_xlim(0, n_spa - 1)
        ax.set_title(f'Det{iDet+1} Ord{iOrder+1}  FWHM={fit_params["fwhm"][1]:.1f}px', fontsize=7)
        ax.set_xlabel('Spatial px', fontsize=6)
        ax.set_ylabel('Mean flux [e-/s]', fontsize=6)
        ax.tick_params(labelsize=5)
        ax.legend(fontsize=4)

plt.suptitle('Position B - 2023-01-01 - 5.A.2: polyfit halo + Moffat residual, physical OLS', fontsize=9)
plt.subplots_adjust(hspace=0.55, wspace=0.45)
plt.show()


#### 1.4 Run PSF extraction for each wavelength channel and Optimal Extraction

In [ ]:
from matplotlib.colors import SymLogNorm
from astropy.visualization import ImageNormalize, ZScaleInterval
from exocrires import plotMatrix

#----Position A ----
# Use a positive value for linthresh
# plot over 3 detectors on one map

#save the mask areas 

mask_areas_A = np.where(np.isnan(spectra_clean_A_phys), 1.0, 0.0)


mask_areas_B = np.where(np.isnan(spectra_clean_B_phys), 1.0, 0.0)

#load data from three detectors into one array
fig, axes = plt.subplots(fit_workflow_A.spectra.shape[1], 2, figsize=(20, 15))
spectra_clean_A_convert = np.zeros_like(spectra_clean_A_phys)

for order in range(spectra_clean_A_phys.shape[1]):

    #Add additional cleaning step for position B for outlier values using sigma clipping

    threshold = 3 # Define a threshold sigma
    #Using the spatial pxiels 5 to 10 and 25 to n_spa to calculate the standard deviation
    std = np.nanstd(np.concatenate((spectra_clean_A_phys[:, order][0:10], spectra_clean_A_phys[:, order][25:n_spa]), axis=0))

    spectra_clean_A_phys[:, order] = np.where((spectra_clean_A_phys[:, order] - np.nanmedian(spectra_clean_A_phys[:,order])) > threshold*std, np.nan, spectra_clean_A_phys[:, order])


    spectra_clean_A_convert[:, order] = np.nan_to_num(spectra_clean_A_phys[:, order], nan=0.0)

    order_all_A = np.zeros((n_spa, 2048*3))
    order_all_A = np.concatenate((spectra_clean_A_convert[0,order], spectra_clean_A_convert[1,order], spectra_clean_A_convert[2,order]), axis=1)


    axes[order, 0].imshow(order_all_A[0:n_spa,:], aspect='auto', norm=SymLogNorm(linthresh=1.0, linscale=1e-1, vmin=max(np.nanmin(order_all_A), 0), vmax=np.nanmax(order_all_A)))
    plt.colorbar(axes[order, 0].images[-1], ax=axes[order, 0])
    axes[order, 0].set_title('Spectra Cleaned Position A - Combined Detectors Order %s'%(23+order))


    #integrate_planet_A = np.array([normalize_profile(np.trapezoid(np.nan_to_num(spectra_clean_A_convert[det,order], nan=0.0), x=np.arange(0,2048,1), axis=1)) for det in range(3)])
    
    integrate_planet_A = np.array([np.trapz(spectra_clean_A_convert[det, order], axis=1) for det in range(3)])
    
    for det in range(3):
        axes[order, 1].plot(integrate_planet_A[det, 0:n_spa], label='Detector %s'%(det+1))

    axes[order, 1].hlines(y=0.0, xmin=0, xmax=n_spa, linestyles='--', color='gray')

    axes[order, 1].set_xlim(5,30)
    axes[order, 1].set_ylim(-0.01, np.nanmax(integrate_planet_A[:,0:35])*1.1)
    axes[order, 1].legend()

plt.subplots_adjust(hspace=0.5, wspace=0.4)
plt.tight_layout()
plt.show()

#----Position B ----
# plot over 3 detectors on one map

fig, axes = plt.subplots(fit_workflow_B.spectra.shape[1], 2, figsize=(20, 15))   
spectra_clean_B_convert = np.zeros_like(spectra_clean_B_phys)

for order in range(spectra_clean_B_phys.shape[1]):
    
    #Add additional cleaning step for position B for outlier values using sigma clipping

    threshold = 3 # Define a threshold sigma
    #Using the spatial pxiels 5 to 10 and 25 to n_spa to calculate the standard deviation
    std = np.nanstd(np.concatenate((spectra_clean_B_phys[:, order][0:10], spectra_clean_B_phys[:, order][25:n_spa]), axis=0))

    spectra_clean_B_phys[:, order] = np.where((spectra_clean_B_phys[:, order] - np.nanmedian(spectra_clean_B_phys[:,order])) > threshold*std, np.nan, spectra_clean_B_phys[:, order])
    #mask negative values as nan
    #spectra_clean_B[:, order] = np.where(spectra_clean_B[:, order] < 0.0, np.nan, spectra_clean_B[:, order])


    spectra_clean_B_convert[:, order] = np.nan_to_num(spectra_clean_B_phys[:, order], nan=0.0)

    order_all_B = np.zeros((n_spa, 2048*3))
    order_all_B = np.concatenate((spectra_clean_B_convert[0,order], spectra_clean_B_convert[1,order], spectra_clean_B_convert[2,order]), axis=1)    

    axes[order, 0].imshow(order_all_B[0:n_spa,:], aspect='auto', norm=SymLogNorm(linthresh=1.0, linscale=1e-1, vmin=max(np.nanmin(order_all_B), 0), vmax=np.nanmax(order_all_B)))
    plt.colorbar(axes[order, 0].images[-1], ax=axes[order, 0])
    axes[order, 0].set_title('Spectra Cleaned Position B - Combined Detectors Order %s'%(23+order))


    #integrate_planet_B = np.array([normalize_profile(np.trapezoid(np.nan_to_num(spectra_clean_B[det, order], nan=0.0), x=np.arange(0,2048,1), axis=1)) for det in range(3)])
    integrate_planet_B = np.array([np.trapz(spectra_clean_B_convert[det, order], axis=1) for det in range(3)])
    
    for det in range(3):
        axes[order, 1].plot(integrate_planet_B[det, 0:n_spa], label='Detector %s'%(det+1))

    axes[order, 1].hlines(y=0.0, xmin=0, xmax=n_spa, linestyles='--', color='gray')
    
    axes[order, 1].set_xlim(5,30)
    axes[order, 1].set_ylim(-0.01, np.nanmax(integrate_planet_B[:,0:n_spa])*1.1)
    axes[order, 1].legend()

plt.subplots_adjust(hspace=0.5, wspace=0.4)
plt.tight_layout()
    
plt.show()

##### i) Run optimal Extraction

In [ ]:
from excalibuhr import utils

half = 5  # half aperture for optimal extraction (8 px total captures ~95% of flux)

# Bug 1 fix: run optimal_extraction on spectra_clean_A_phys — the polynomial-subtracted
# companion residual in physical units (e-/s), computed above (method 5.A.2).
#
# spectra_clean_A_phys[:, i] = spectra_clean_A[:, i] * col_sum_i
#   = spec2d[:, i] - model_background_poly * col_sum_i
# i.e. raw counts minus the scaled polynomial stellar halo — the PSF is removed
# while keeping physical units so the Poisson term in optimal_extraction is correct.
#
# Sky variance is computed from the same physical PSF-subtracted array at sky rows.

spectra_clean_A_phys = np.nan_to_num(spectra_clean_A_phys, nan=0.0)
spectra_clean_B_phys = np.nan_to_num(spectra_clean_B_phys, nan=0.0)

extracted_spectra_A = np.zeros((3, 5, spectra_clean_A_phys.shape[3]))
extracted_spectra_A_err = np.zeros((3, 5, spectra_clean_A_phys.shape[3]))

fig, axes = plt.subplots(5, 1, figsize=(20, 10))
for det in range(3):
    for order in range(5):
        D_phys = spectra_clean_A_phys[det, order, :, :]  # (45, nWave), physical units

        # Sky variance from physical PSF-subtracted sky rows (Position A: rows 0:5 and 25:30)
        _sky_A = np.concatenate([
            spectra_clean_A_phys[det, order, 0:5, :],
            spectra_clean_A_phys[det, order, 25:30, :]], axis=0)
        _sky_var_A = np.var(_sky_A, axis=0, ddof=1)
        Var_phys = np.tile(_sky_var_A, (D_phys.shape[0], 1))

        center = int(moffat_params_A[det, order]['x0'][1])

        opt_extraction_A = utils.optimal_extraction(
            D_full=D_phys[center-half:center+half, :].T,
            V_full=Var_phys[center-half:center+half, :].T,
            aper_half=half, filter_width=half*2+1,
            bpm_full=mask_areas_A[det, order, center-half:center+half, :].T,
            obj_cen=half, badpix_clip=3, max_iter=100)

        extracted_spectra_A[det, order, :] = opt_extraction_A[0]
        extracted_spectra_A_err[det, order, :] = opt_extraction_A[1]

        axes[order].plot(wave_data[det, order], opt_extraction_A[0], linewidth=0.6, alpha=0.7)
        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]),
                           xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')

plt.tight_layout()
plt.show()

extracted_spectra_B = np.zeros((3, 5, spectra_clean_B_phys.shape[3]))
extracted_spectra_B_err = np.zeros((3, 5, spectra_clean_B_phys.shape[3]))

fig, axes = plt.subplots(5, 1, figsize=(20, 10))
for det in range(3):
    for order in range(5):
        D_phys_B = spectra_clean_B_phys[det, order, :, :]  # (45, nWave), physical units

        # Sky variance from physical PSF-subtracted sky rows (Position B: rows 5:10 and 26:30)
        _sky_B = np.concatenate([
            spectra_clean_B_phys[det, order, 5:10, :],
            spectra_clean_B_phys[det, order, 26:30, :]], axis=0)
        _sky_var_B = np.var(_sky_B, axis=0, ddof=1)
        Var_phys_B = np.tile(_sky_var_B, (D_phys_B.shape[0], 1))

        center = int(moffat_params_B[det, order]['x0'][1])

        opt_extraction_B = utils.optimal_extraction(
            D_full=D_phys_B[center-half:center+half, :].T,
            V_full=Var_phys_B[center-half:center+half, :].T,
            aper_half=half, filter_width=half*2+1,
            bpm_full=mask_areas_B[det, order, center-half:center+half, :].T,
            obj_cen=half, badpix_clip=3, max_iter=100)

        extracted_spectra_B[det, order, :] = opt_extraction_B[0]
        extracted_spectra_B_err[det, order, :] = opt_extraction_B[1]

        axes[order].plot(wave_data[det, order], opt_extraction_B[0], linewidth=0.6, alpha=0.7)
        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]),
                           xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')

plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------------------------------------------------
# Blaze correction (applied after optimal extraction, before sigma-clipping)
#
# The blaze function is the wavelength-dependent grating efficiency that
# creates a smooth broad envelope within each spectral order.  Excalibuhr
# corrects it for its own 1D output via f_opt/blaze[o] (utils.py line 1565)
# but NOT for the Extr2D product (D_sub) that this pipeline uses.  Applying
# the correction here after optimal extraction is mathematically equivalent
# to applying it in 2D before extraction because the blaze is spatially
# uniform (same scalar for every spatial row at a given wavelength column).
#
# BLAZE_K2166.fits shape: (3 detectors, 7 orders, 2048 px).
# Orders 0-4 in the blaze array correspond 1-to-1 with the 5 science orders
# used in this notebook (same indexing as wave_data and Extr2D).
# -----------------------------------------------------------------------
from astropy.io import fits as _fits

_blaze_raw = _fits.getdata(
    workpath + '/cal/BLAZE_K2166.fits', 1)  # shape (3, 7, 2048)
# Replace zeros/negatives with NaN to avoid division artefacts at order edges
_blaze = np.where(_blaze_raw > 0, _blaze_raw, np.nan)

for det in range(3):
    for order in range(5):
        b = _blaze[det, order, :]                      # shape (2048,)
        extracted_spectra_A[det, order, :]     /= b
        extracted_spectra_A_err[det, order, :] /= b
        extracted_spectra_B[det, order, :]     /= b
        extracted_spectra_B_err[det, order, :] /= b

print('Blaze correction applied. Blaze range det=2, order=3 (most affected):',
      f'{np.nanmin(_blaze[2, 3]):.0f} – {np.nanmax(_blaze[2, 3]):.0f} counts')


##### ii-A) Classic Sigma-Clipper

In [ ]:
#remove the cosmic lines/bad pixels again using sigma clipping on the extracted spectra
n_iterations = 100
threshold = 3

extracted_spectra_A_clip = np.copy(extracted_spectra_A)
extracted_spectra_B_clip = np.copy(extracted_spectra_B)

for iteration in range(n_iterations): #number of iterations
    for det in range(3):
        for order in range(5):
            #threshold = 6 # Define a threshold sigma
            std = np.nanstd(extracted_spectra_A[det, order])
            median = np.nanmedian(extracted_spectra_A[det, order])
            extracted_spectra_A_clip[det, order] = np.where(np.abs(extracted_spectra_A[det, order] - median) > threshold*std, median, extracted_spectra_A[det, order])

            std = np.nanstd(extracted_spectra_B[det, order])
            median = np.nanmedian(extracted_spectra_B[det, order])
            extracted_spectra_B_clip[det, order] = np.where(np.abs(extracted_spectra_B[det, order] - median) > threshold*std, median, extracted_spectra_B[det, order])

#plot the extracted spectra after cosmic ray removal
fig, axes = plt.subplots(5, 1, figsize=(20, 10))

for det in range(3):
    for order in range(5):
        
        axes[order].plot(wave_data[det,order], extracted_spectra_A[det, order], linewidth=0.6, alpha=1, label='Original Detector %s'%(det+1), color='gray')
        axes[order].plot(wave_data[det,order], extracted_spectra_A_clip[det, order], linewidth=0.6, alpha=0.7, label='Detector %s'%(det+1))

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
        #axes[order].set_ylim(-0.1, 0.6)
        axes[order].set_title('Extracted Spectra Position A - Order %s'%(23+order))
        axes[order].legend()

plt.tight_layout()
plt.show()

#plot the extracted spectra after cosmic ray removal
fig, axes = plt.subplots(5, 1, figsize=(20, 10))

for det in range(3):
    for order in range(5):
        
        axes[order].plot(wave_data[det,order], extracted_spectra_B[det, order], linewidth=0.6, alpha=1, label='Original Detector %s'%(det+1), color='gray')
        axes[order].plot(wave_data[det,order], extracted_spectra_B_clip[det, order], linewidth=0.6, alpha=0.7, label='Detector %s'%(det+1))

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
        #axes[order].set_ylim(-0.1, 0.6)
        axes[order].set_title('Extracted Spectra Position B - Order %s'%(23+order))
        axes[order].legend()
        
plt.tight_layout()
plt.show()

##### ii-B) Low-pass filter

In [ ]:
#remove the cosmic lines/bad pixels again using low-pass filter on the extracted spectra
n_iterations = 100
threshold = 3

extracted_spectra_A_clip = np.copy(extracted_spectra_A)
extracted_spectra_B_clip = np.copy(extracted_spectra_B)

continuum_A = np.zeros_like(extracted_spectra_A)
continuum_B = np.zeros_like(extracted_spectra_B)

for iteration in range(n_iterations): #number of iterations
    for det in range(3):
        for order in range(5):

            error_A = extracted_spectra_A_err[det, order]

            mask_isfinite_A = np.isfinite(extracted_spectra_A[det, order])

            continuum_A[det, order] = savgol_filter(extracted_spectra_A[det, order][mask_isfinite_A], window_length=301, polyorder=2)
            extracted_spectra_A_clip[det, order] = np.where(np.abs(extracted_spectra_A[det, order] - continuum_A[det, order]) > threshold*error_A, continuum_A[det, order], extracted_spectra_A[det, order])

            error_B = extracted_spectra_B_err[det, order]

            mask_isfinite_B = np.isfinite(extracted_spectra_B[det, order])

            continuum_B[det, order] = savgol_filter(extracted_spectra_B[det, order][mask_isfinite_B], window_length=301, polyorder=2)
            extracted_spectra_B_clip[det, order] = np.where(np.abs(extracted_spectra_B[det, order] - continuum_B[det, order]) > threshold*error_B, continuum_B[det, order], extracted_spectra_B[det, order])
#plot the extracted spectra after cosmic ray removal
fig, axes = plt.subplots(5, 1, figsize=(20, 10))

for det in range(3):
    for order in range(5):
        axes[order].plot(wave_data[det,order], extracted_spectra_A[det, order], linewidth=0.6, alpha=1, label='Detector %s'%(det+1),  color='darkgray')
        axes[order].plot(wave_data[det,order], extracted_spectra_A_clip[det, order], linewidth=0.6, alpha=0.7, label='Detector %s - Filtered'%(det+1))
        axes[order].plot(wave_data[det,order], continuum_A[det, order], linewidth=0.6, alpha=0.7, color='red')

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
        axes[order].set_ylim(-0.1, 0.6)
        axes[order].set_title('Extracted Spectra Position A - Order %s'%(23+order))
        axes[order].legend()

plt.tight_layout()
plt.show()

#plot the extracted spectra after cosmic ray removal
fig, axes = plt.subplots(5, 1, figsize=(20, 10))

for det in range(3):
    for order in range(5):
        axes[order].plot(wave_data[det,order], extracted_spectra_B[det, order], linewidth=0.6, alpha=1, label='Detector %s'%(det+1), color='darkgray')
        axes[order].plot(wave_data[det,order], extracted_spectra_B_clip[det, order], linewidth=0.6, alpha=0.7, label='Detector %s - Filtered'%(det+1))
        axes[order].plot(wave_data[det,order], continuum_B[det, order], linewidth=0.6, alpha=0.7, color='red')

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
        axes[order].set_ylim(-0.1, 0.6)
        axes[order].set_title('Extracted Spectra Position B - Order %s'%(23+order))
        axes[order].legend()
        
plt.tight_layout()
plt.show()

##### ii-C) Fourier-Based high-frequency spekle masking

In [ ]:
# Define a function for Fourier-based noise masking
def fourier_noise_masking(data, threshold=0.01, n_iterations=5):
    for ite in range(n_iterations):
        data = np.nan_to_num(data, nan=0.0)
        # Perform FFT
        fft_data = np.fft.fft(data)
        frequencies = np.fft.fftfreq(len(data))

        # Identify high-frequency components
        high_freq_mask = np.abs(frequencies) > threshold

        # Suppress high-frequency components
        fft_data[high_freq_mask] = 0

        # Perform inverse FFT
        data = np.fft.ifft(fft_data).real

    filtered_data = data
    return filtered_data

extracted_spectra_A_filter = np.zeros_like(extracted_spectra_A)
extracted_spectra_B_filter = np.zeros_like(extracted_spectra_B)

# Apply the Fourier-based noise masking to the spectral data
for det in range(3):
    for order in range(5):
        extracted_spectra_A_filter[det, order] = fourier_noise_masking(extracted_spectra_A[det, order], threshold=0.1)
        extracted_spectra_B_filter[det, order] = fourier_noise_masking(extracted_spectra_B[det, order], threshold=0.1)


# Plot the original and filtered spectra for comparison
fig, axes = plt.subplots(5, 1, figsize=(20, 10))
for det in range(3):
    for order in range(5):
        axes[order].plot(wave_data[det,order], extracted_spectra_A[det, order], label='Original Detector %s'%(det+1), color='gray', alpha=0.7)
        axes[order].plot(wave_data[det,order], extracted_spectra_A_filter[det, order], label='Filtered Detector %s'%(det+1), alpha=0.7)

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
        #axes[order].set_ylim(-0.1, 0.6)
        axes[order].set_title('Fourier Filtered Extracted Spectra Position A - Order %s'%(23+order))
        axes[order].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(5, 1, figsize=(20, 10))
for det in range(3):
    for order in range(5):
        axes[order].plot(wave_data[det,order], extracted_spectra_B[det, order], label='Original Detector %s'%(det+1), color='gray', alpha=0.7)
        axes[order].plot(wave_data[det,order], extracted_spectra_B_filter[det, order], label='Filtered Detector %s'%(det+1), alpha=0.7)

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
        #axes[order].set_ylim(-0.1, 0.6)
        axes[order].set_title('Fourier Filtered Extracted Spectra Position B - Order %s'%(23+order))
        axes[order].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save the extracted spectra (optimal extraction output, no sigma clipping applied)
# Using 5.A.2 two-kernel Moffat fit + optimal extraction on raw counts (Bug 1 fix)

np.save(workpath + '/extracted_spectra_position_A_sigmaclipper_0417_0101.npy', extracted_spectra_A_clip)
np.save(workpath + '/extracted_spectra_position_B_sigmaclipper_0417_0101.npy', extracted_spectra_B_clip)

np.save(workpath + '/extracted_spectra_position_A_err_0417_0101.npy', extracted_spectra_A_err)
np.save(workpath + '/extracted_spectra_position_B_err_0417_0101.npy', extracted_spectra_B_err)


In [ ]:
#bin the extracted spectra to R=1000
#smoothen the spectrum with a low resolution filter to make the absorption lines more visible
def smooth_spectrum(spectrum, window_length=50, polyorder=2):
    """Smooth the spectrum using a Savitzky-Golay filter."""
    #spectrum = np.nan_to_num(spectrum, nan=0.0)
    smoothed_spectrum = savgol_filter(spectrum, window_length=window_length, polyorder=polyorder)
    return smoothed_spectrum

prep_planet = Spectra_Preparator (folder_name='2022-12-31', T=2300, G=4.0, Z=-0.0, path=workpath)

v= 31 

planet_model=np.load('/data2/peng/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))


fig, axes = plt.subplots(5, 1, figsize=(20, 20))

mask_nan_A = np.isnan(extracted_spectra_A)
mask_nan_B = np.isnan(extracted_spectra_B)


for order in range(5):
    for det in range(3):
        planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])
        planet_model_spec -= savgol_filter (planet_model_spec, window_length=300, polyorder=2)  

        extracted_spectra_A[det, order][~mask_nan_A[det, order]] = extracted_spectra_A[det, order][~mask_nan_A[det, order]]
        extracted_spectra_B[det, order][~mask_nan_B[det, order]] = extracted_spectra_B[det, order][~mask_nan_B[det, order]]

        extracted_spectra_A[det, order][~mask_nan_A[det, order]] -= savgol_filter (extracted_spectra_A[det, order][~mask_nan_A[det, order]], window_length=300, polyorder=2)
        extracted_spectra_B[det, order][~mask_nan_B[det, order]] -= savgol_filter (extracted_spectra_B[det, order][~mask_nan_B[det, order]], window_length=300, polyorder=2)


        smoothed_A = smooth_spectrum(extracted_spectra_A[det, order][~mask_nan_A[det, order]], window_length=100, polyorder=2)
        smoothed_B = smooth_spectrum(extracted_spectra_B[det, order][~mask_nan_B[det, order]], window_length=100, polyorder=2)

        smoothed_model= smooth_spectrum(planet_model_spec[~mask_nan_A[det, order]], window_length=100, polyorder=2)

        #plot original and smoothed spectrum

        #axes[order].plot(wave_data[det, order][~mask_nan_A[det, order]], extracted_spectra_A[det, order][~mask_nan_A[det, order]], linestyle=':')
        #axes[order].plot(wave_data[det, order][~mask_nan_B[det, order]], extracted_spectra_B[det, order][~mask_nan_B[det, order]], linestyle='--')
        axes[order].plot(wave_data[det, order][~mask_nan_A[det, order]], smoothed_A, label='Smoothed A - Det %s'%(det+1), linewidth=3, linestyle='-')
        axes[order].plot(wave_data[det, order][~mask_nan_B[det, order]], smoothed_B, label='Smoothed B - Det %s'%(det+1), linewidth=3, linestyle='-')
        axes[order].plot(wave_data[det, order][~mask_nan_A[det, order]], smoothed_model, label='Planet Model - Det %s'%(det+1), linewidth=2.5, linestyle=':')
        

        axes[order].hlines(y=0.0, xmin=np.nanmin(wave_data[det, order]), xmax=np.nanmax(wave_data[det, order]), linestyles='--', color='gray')
    
    axes[order].set_title('Extracted and Smoothed Spectra - Order %s'%(order+1))
    #axes[order].set_ylim(-0.15, 0.2)
    axes[order].set_xlabel('Wavelength (micron)')
    axes[order].set_ylabel('Flux (arbitrary units)')
    #axes[order].legend(fontsize=6)

plt.tight_layout()
plt.show()

## 2. Extract the spectrum following Rico et al. 2024

#### 2.1 Prepare the stellar model spectrum


In [ ]:
#-----Prepare the stellar spectrum-----
band=-3
wave_range=info.lambda_range['wave'][band]
order_range=info.lambda_range['order'][band]

#-----Initialize the preparetor -----
prep_star = Spectra_Preparator (folder_name='2023-01-01', T=3700, G=4.0, Z=-0.0, path='/Users/richard/project_DH_Tau_B/')

#-----generate the broadened spectra-----
path_throughput = ['/Users/richard/project_EXOCRIRES/throughput/SPHERE_IRDIS_B_Ks.dat', '/Users/richard/project_EXOCRIRES/throughput/PARANAL_ERIS_Lp.dat']
ld_coefficients=prep_star.compute_limb_darkening(wavelength_range=info.lambda_range['wave'][-3:-1], throughput_files=path_throughput)

wave_phoe, flux_phoe = prep_star.load_phoenix_spectrum()

wave_phoe *= 1e-10

wave_lsf, flux_lsf_broaden=prep_star.apply_broadening(wave=wave_phoe, flux=flux_phoe, lam_range=wave_range, ld_coeff=float(ld_coefficients[0]), rotation=7.1, step_size=1e-11)

#-----Inject the RV-----
v=16.38 #km/s R.V. for DH Tau A

wave_shift, flux_shift = prep_star.radial_velocity_shift(wave=wave_lsf, flux=flux_lsf_broaden, velocity=v)

sav_path = prep_star.path +'spectra_input'

#-----Cut and save the spectra into different orders-----
if os.path.exists (sav_path) != True:

    os.makedirs(sav_path, exist_ok=True)

filenames_spectra=['/Star_%s_%s_%s'%(prep_star.T, prep_star.G, prep_star.Z)+'_order_%s.dat'%i for i in order_range]
prep_star.spectrum_cutter(lam_range=wave_range, wave=wave_shift, flux=flux_shift, saving_path=sav_path, filenames=filenames_spectra)

#Prepare the stellar spectra into the shape of observation matrix

wave_cal = fits.open('/Users/richard/project_DH_Tau_B/2023-01-01/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')

wave_data=wave_cal[1].data

phoenix_matrix = prep_star.spectrum_cutter(wave=wave_shift*1e9, flux=flux_shift, saving_file=True, lam_range=wave_range,
filenames='/Star_%s_%s_%s.npy'%(prep_star.T, prep_star.G, prep_star.Z), ignore_detector=False, Wave_solution=wave_data,
saving_path=prep_star.path+'spectra_input')


plt.plot(wave_phoe, flux_phoe, label='original spectrum', alpha=0.4)
plt.plot(wave_lsf, flux_lsf_broaden, label='broadened spectrum', alpha=0.8)
#plt.plot(wave_shift, flux_shift, label='broadened spectrum with RV shift', alpha=0.8)
plt.plot()

plt.xlim(2.25e-6, 2.30e-6)
plt.ylim(0, 2e13)

plt.xlabel('wavelength (m)')
plt.ylabel('flux (erg/s/cm^2/cm)')

plt.legend()
plt.show()


### 2.2 Prepare the first-look planet spectrm

In [ ]:
#Prepare the planet spectrum
band=-3
wave_range=info.lambda_range['wave'][band:(band+1)]
order_range=info.lambda_range['order'][band:(band+1)]

prep_planet = Spectra_Preparator (folder_name='2023-01-01', T=2300, G=4.0, Z=-0.0, path='/Users/richard/project_DH_Tau_B/')

Flux_planet = np.load(prep_planet.path+prep_planet.folder_name+'/f_pRT_t2300_g100nc_m0.0.npy')
Wave_planet = np.load(prep_planet.path+prep_planet.folder_name+'/wave_pRT.npy')



ld_coefficients=prep_planet.compute_limb_darkening(wavelength_range=wave_range, \
                                                 throughput_files=[path_throughput[0]],\
                                                wave=Wave_planet*1e4, flux=Flux_planet, Mode='custom')



wave_lsf, flux_lsf_broaden=prep_planet.apply_broadening(wave=Wave_planet*1e-6, flux=Flux_planet,\
                                                         lam_range=wave_range[0], ld_coeff=ld_coefficients[0],\
                                                         rotation=5.7, step_size=1e-11)

#-----Inject the RV-----
for v in tqdm.tqdm(np.arange(-100, 100, 1)):

    wave_shift, flux_shift = prep_planet.radial_velocity_shift(wave=wave_lsf, flux=flux_lsf_broaden, velocity=v)

    sav_path = prep_planet.path +'spectra_input'

    #-----Cut and save the spectra into different orders-----
    if os.path.exists (sav_path) != True:

        os.makedirs(sav_path, exist_ok=True)

    filenames_spectra=['/Planet_%s_%s_%s_%s'%(prep_planet.T, prep_planet.G, prep_planet.Z, v)+'_order_%s.dat'%i for i in order_range[0]]
    prep_planet.spectrum_cutter(lam_range=wave_range[0], wave=wave_shift, flux=flux_shift, saving_path=sav_path, filenames=filenames_spectra)
    
    file='/planet_%s_%s_%s_%s.dat'%(prep_planet.T, prep_planet.G, prep_planet.Z, v)

    save_file = sav_path +file

    prep_planet.save_spectrum(wave=wave_shift, flux=flux_shift, filename=save_file)


In [ ]:
v=31 #km/s
#Prepare the planet spectra into the shape of observation matrix
wave_cal = fits.open(f'/Users/richard/project_DH_Tau_B/{prep_planet.folder_name}/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')

spec_shift = np.loadtxt(sav_path + '/Planet_%s_%s_%s_%s.dat'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))
wave_shift = spec_shift[0]
flux_shift = spec_shift[1]

wave_data=wave_cal[1].data

planet_matrix = prep_planet.spectrum_cutter(wave=wave_shift*1e9, flux=flux_shift, saving_file=True, lam_range=wave_range,
filenames='/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v), ignore_detector=False, Wave_solution=wave_data,
saving_path=prep_planet.path+'spectra_input')

spec_shift = np.loadtxt(sav_path + '/Planet_%s_%s_%s_%s.dat'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))
wave_shift = spec_shift[0]
flux_shift = spec_shift[1]

plt.plot(Wave_planet*1e-6, Flux_planet, label='original spectrum', alpha=0.4)
plt.plot(wave_lsf, flux_lsf_broaden, label='broadened spectrum', alpha=0.8)
#plt.plot(wave_shift, flux_shift, label='broadened spectrum with RV shift', alpha=0.8)
plt.plot()

plt.xlim(2.30e-6, 2.32e-6)
plt.ylim(0, 2e13)

plt.xlabel('wavelength (m)')
plt.ylabel('flux (erg/s/cm^2/cm)')

plt.legend()
plt.show()

In [ ]:
# Prepare the molecular tempaltes for the planet
for mol in ['ch4_only', 'co_only', 'h2o_pokazatel_only', 'no_ch4', 'no_co', 'no_h2o_pokazatel']:

    Flux_planet = np.load(prep_planet.path+prep_planet.folder_name+'/f_pRT_%s_t2000_g100nc_m0.0.npy'%mol)
    Wave_planet = np.load(prep_planet.path+prep_planet.folder_name+'/wave_pRT.npy')

    wave_lsf, flux_lsf_broaden=prep_planet.apply_broadening(wave=Wave_planet*1e-6, flux=Flux_planet, lam_range=wave_range[0], ld_coeff=ld_coefficients[0],\
                                                             rotation=5.7, step_size=1e-11)


    #-----Inject the RV-----
    for v in tqdm.tqdm(np.arange(-100, 100, 1)):

        wave_shift, flux_shift = prep_planet.radial_velocity_shift(wave=wave_lsf, flux=flux_lsf_broaden, velocity=v)

        sav_path = prep_planet.path +'spectra_input'

        #-----Cut and save the spectra into different orders-----
        if os.path.exists (sav_path) != True:

            os.makedirs(sav_path, exist_ok=True)

        filenames_spectra=['/Planet_%s_%s_%s_%s_%s'%(prep_planet.T, prep_planet.G, prep_planet.Z, v, mol)+'_order_%s.dat'%i for i in order_range[0]]
        prep_planet.spectrum_cutter(lam_range=wave_range[0], wave=wave_shift, flux=flux_shift, saving_path=sav_path, filenames=filenames_spectra)

        file='/planet_%s_%s_%s_%s_%s.dat'%(prep_planet.T, prep_planet.G, prep_planet.Z, v, mol)

        save_file = sav_path +file

        prep_planet.save_spectrum(wave=wave_shift, flux=flux_shift, filename=save_file)

### 2.3 Modify the data structure with the shape of observation matrices

In [ ]:
#load the wavelength data and the transmission templates



trans = fits.open(f'/Users/richard/project_DH_Tau_B/{prep_planet.folder_name}/cal/TRANSM_SPEC.fits')

trans_data = trans[1].data

plt.plot(trans_data[:,0], trans_data[:,1], linewidth=0.5)
plt.show()



#Prepare the transmission spectra into the shape of observation matrix
trans_matrix = prep_star.spectrum_cutter(wave=trans_data[:,0], flux=trans_data[:,1], saving_file=True, lam_range=wave_range,
filenames='/Trans.npy', ignore_detector=False, Wave_solution=wave_data,
saving_path=prep_star.path+'spectra_input')





### 2.4 Mask the low transmission areas and bad pixels

In [ ]:
#--------------------

fit_workflow_A = Spectra_fitting(spectra2D=flux2d_A, transmission_matrix=trans_matrix, window_length=300, polyorder=2)
spectra_masked_A=fit_workflow_A.Mask_spec(atm_threshold=0.7,  sigma_threshold=3, Mask_bad_pxiel=True)

fig, axes = plt.subplots(7,3, figsize=(12,18))

plt.subplots_adjust(hspace=0.3)
plt.suptitle('Positon A, Night 1', fontsize=10)
for i in range(3):
    for j in range(7):
        ax=axes[j,i]
        wl=wave_data[i,j]
        img=spectra_masked_A[i][j]
        map=ax.imshow(img, aspect='auto',norm=SymLogNorm(5, 500), extent=[wl.min(), wl.max(), 0, img.shape[0]])

        ax.set_title('Order %s, Detector %s'%((23+j), (i+1)), fontsize=6) 

        # X axis: smaller font, ~3–4 ticks, integer values
        ax.tick_params(axis="x", labelsize=8)
        ax.locator_params(axis="x", nbins=4, integer=True)
        
        # Label only outer axes to reduce clutter
        if j == 6:  # bottom row
            ax.set_xlabel("Wavelength", fontsize=9)

        if i == 0:  # first column
            ax.set_ylabel("Spatial position", fontsize=9)


        plt.colorbar(map)

#plt.tight_layout()       
plt.show()

#-----------------------

flux2d_B_pad = np.zeros(shape=flux2d_A.shape)

for i in range(flux2d_B_pad.shape[0]):

    for j in range (flux2d_B_pad.shape[1]):

        arr=flux2d_B[i][j].shape[0]

        flux2d_B_pad[i][j][0:arr] = flux2d_B[i][j]

fit_workflow_B = Spectra_fitting(spectra2D=flux2d_B_pad, transmission_matrix=trans_matrix, window_length=300, polyorder=2)
spectra_masked_B=fit_workflow_B.Mask_spec(atm_threshold=0.6,  sigma_threshold=1, Mask_bad_pxiel=True)

fig, axes = plt.subplots(7,3, figsize=(12,18))

plt.subplots_adjust(hspace=0.3)
plt.suptitle('Positon B, Night 1', fontsize=10)
for i in range(3):
    for j in range(7):
        ax=axes[j,i]
        wl=wave_data[i,j]
        img=spectra_masked_B[i][j]
        map=ax.imshow(img, aspect='auto',norm=SymLogNorm(5, 500), extent=[wl.min(), wl.max(), 0, img.shape[0]])

        ax.set_title('Order %s, Detector %s'%((23+j), (i+1)), fontsize=6) 


        # X axis: smaller font, ~3–4 ticks, integer values
        ax.tick_params(axis="x", labelsize=8)
        ax.locator_params(axis="x", nbins=4, integer=True)
        
        # Label only outer axes to reduce clutter
        if j == 6:  # bottom row
            ax.set_xlabel("Wavelength", fontsize=9)
    
        if i == 0:  # first column
            ax.set_ylabel("Spatial position", fontsize=9)

        plt.colorbar(map)

#plt.tight_layout()       
plt.show()


### 2.5 Spectral Fitting

#### 2.5.0 Generate Master templates and calculate the integrated flux ratio

In [ ]:
master_matrix_A = fit_workflow_A.stellar_master_spec(wave=wave_data, cut_off=2)

master_matrix_B = fit_workflow_B.stellar_master_spec(wave=wave_data, cut_off=2)

#combine_A_B= fit_workflow_A.spectra[:,0:5]+fit_workflow_B.spectra[:,0:5]

fit_workflow_A.spectra=fit_workflow_A.spectra[:,0:5]
fit_workflow_B.spectra=fit_workflow_B.spectra[:,0:5]



#### 2.5.1 Fit pair A-B

In [ ]:
v= 31
planet_model=np.load(prep_planet.path+'/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))
window=300
polyorders = 2

fit_matrix_planet, fit_matrix_star, fit_alpha_star, fit_alpha_planet, fit_model, popt_list=fit_workflow_A.multiplicative_factor_fit(planet_model[:, 0:5], window_len=window,
                                                                                                                                     Polyorder=polyorders, 
                                                                                                                                     master=master_matrix_A[:, 0:5])

In [ ]:
# run several tests to check the results
po=60
i=1
x=1

#multi_factor=(fit_alpha_star[-3,:, 0:2046].T*popt_list[-3, :, 0, 0])

#plt.plot(multi_factor[1000],label=r'$c_s\alpha (\lambda)$')
plt.plot(fit_alpha_star[x,i,:,1000], label=r'$ \alpha $')
#plt.vlines(x=17, ymin=0, ymax=3e-2, ls='--')
#plt.vlines(x=27, ymin=0, ymax=3e-2, ls='--')
#plt.ylim(0.647,0.649)
#plt.xlim(1000,1010)
plt.legend()
plt.show()


plt.plot(popt_list[x, i, :, 1], label=['cp'])
#plt.hlines(y=1, xmin=0, xmax=35, ls='--', colors='gray')
#plt.vlines(x=(60+po), ymin=0, ymax=0.02, ls='--')
plt.xlabel('Spatial axis')
plt.ylabel('Fitted coefficient value')
#plt.yscale('symlog')
plt.legend()
plt.show()

mask=np.array(master_matrix_A[x][i] != 0)
test_master=savgol_filter(master_matrix_A[x][i][mask], window, polyorder=2)
plt.plot(master_matrix_A[x][i], alpha=0.3)
plt.plot(test_master, label='lowpass')

plt.legend()
plt.show()

plt.plot(fit_alpha_star[x, i, 60])
#plt.ylim(0.64, 0.66)

In [ ]:
order=1
det=0
stack = fit_workflow_A.spectra
central_pix=60
spatial_pix=int(21)

planet_model=np.load(prep_planet.path+'/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))

planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])

planet_model_spec -= savgol_filter (planet_model_spec, window_length=300, polyorder=2)

#planet_model_spec -= np.nanmedian(planet_model_spec)

po = int (central_pix-spatial_pix)

mask_data = np.isfinite(stack[det, order, central_pix])
mask_wave = np.array(wave_data[det,order]!=0)

stack[det, order, central_pix][~mask_data] = np.nan


plt.plot(wave_data[det,order][mask_wave&mask_data], stack[det, order, central_pix][mask_wave&mask_data], label='original data')
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix_star[det, order, central_pix][mask_wave&mask_data], label='stellar fit')

plt.xlabel('wavelength (m)')
plt.ylabel('flux (counts)')
plt.title('Pix %s'%central_pix)

plt.legend()
plt.show()

plt.plot(wave_data[det,order][mask_wave&mask_data], stack[det, order,spatial_pix][mask_wave&mask_data],label='original data')
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix_star[det, order,spatial_pix][mask_wave&mask_data], label='stellar fit')


plt.title('Pix %s'%spatial_pix)

plt.xlabel('wavelength (m)')
plt.ylabel('flux (counts)')

plt.legend()
plt.show()


fit_matrix=fit_matrix_star+fit_matrix_planet

mask_fit=np.array(fit_matrix_star==0.)

fit_matrix_star[mask_fit]=np.nan

mask_fit=np.array(fit_matrix_planet==0.)

fit_matrix_planet[mask_fit]=np.nan






#plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, stack[order,spatial_pix][mask_wave&mask_data],label='original data')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_star[det, order,spatial_pix][mask_wave&mask_data], label='stellar fit')

#plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_star[order,spatial_pix][mask_wave&mask_data], label='stellar fit')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_planet[det, order,spatial_pix][mask_wave&mask_data], label='planet fit')

#plt.xlim(22500, 22520)
#plt.ylim(-1e3, 1.5e5)

plt.title(' Planet position')

plt.xlabel('wavelength ($\AA$)')
plt.ylabel('flux (counts)')

plt.yscale('symlog')

plt.legend()
plt.show()



#plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_star[order,spatial_pix][mask_wave&mask_data], label='stellar fit')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, stack[det, order,spatial_pix][mask_wave&mask_data], label='Mock observation data')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix[det, order,spatial_pix][mask_wave&mask_data], label='stellar fit + planet-fit')

#plt.xlim(22500, 22520)
#plt.ylim(-1e3, 1.5e5)

plt.title(' Planet position')

plt.xlabel('wavelength ($\AA$)')
plt.ylabel('flux (counts)')

#plt.yscale('symlog')

plt.legend()
plt.show()

subtract=stack-fit_matrix_star

subtract_spec = (subtract[det, order,spatial_pix][mask_wave&mask_data]/np.nanmax(subtract[det, order,spatial_pix][mask_wave&mask_data]) - np.nanmedian(subtract[det, order,spatial_pix][mask_wave&mask_data]))
#subtract_spec -= savgol_filter(subtract_spec, window_length=300, polyorder=2)

fit_spec = (fit_matrix_planet[det, order, spatial_pix][mask_wave&mask_data] - np.nanmedian(fit_matrix_planet[det, order, spatial_pix][mask_wave&mask_data]))
#fit_spec -= savgol_filter(fit_spec, window_length=300, polyorder=2)

plt.plot(wave_data[det,order][mask_wave&mask_data], subtract_spec, label='original data - stellar fit', linewidth=0.2)

#plt.plot(wave_data[det,order][mask_wave&mask_data],fit_spec ,label='planet fit')

plt.plot(wave_data[det,order], planet_model_spec,label='planet model')

plt.title('Pix %s'%spatial_pix)

plt.xlim(2332, 2336)

plt.xlabel('wavelength axis')
plt.ylabel('flux')

plt.legend()
plt.show()


#plt.plot(wave_data[det,order][mask_wave&mask_data], stack[order, spatial_pix][mask_wave&mask_data], label='mock observation at planet position')
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix[det, order, spatial_pix][mask_wave&mask_data], label='planet position', alpha=0.9)
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix[det, order, central_pix+po][mask_wave&mask_data], label='reference position', alpha=0.7)


plt.xlabel('wavelength (m)')
plt.ylabel('flux (counts)')

plt.title('Fit Matrix $\mathbf{M_\psi}$')

plt.legend()
plt.show()



plt.plot(wave_data[det,order][mask_wave&mask_data], fit_spec, label='observation')
plt.plot(wave_data[det,order][mask_wave&mask_data], planet_model_spec[mask_wave&mask_data], label='Planet model', alpha=0.6)

plt.xlabel('wavelength axis')
plt.ylabel('flux (counts)')
plt.title("Fitted planet flux $d_p\'$")
plt.legend()

plt.show()

#-----------bin the spectrum to increase S/N ratio-----
def bin_spectrum(wavelength, spectrum, bin_size=40):
    """Bin the spectrum by averaging over specified bin size."""
    n_bins = len(spectrum) // bin_size
    binned_wavelength = np.nanmean(wavelength[:n_bins*bin_size].reshape(-1, bin_size), axis=1)
    binned_spectrum = np.nanmean(spectrum[:n_bins*bin_size].reshape(-1, bin_size), axis=1)
    return binned_wavelength, binned_spectrum

plt.figure(figsize=(12,5))
for det in range(3):

    planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])
    planet_model_spec -= savgol_filter(planet_model_spec, window_length=300, polyorder=2)
    
    smoothed_wl, smoothed_spec_model = bin_spectrum(wave_data[det,order], planet_model_spec)

    subtract_spec = (subtract[det, order,spatial_pix]/np.nanmax(subtract[det, order,spatial_pix]) - np.nanmedian(subtract[det, order,spatial_pix]))
    
    smoothed_wl, smoothed_spec_subtract = bin_spectrum(wave_data[det,order], subtract_spec)

    #factor = np.median(smoothed_spec_subtract)/np.median(smoothed_spec_model)

    #smoothed_spec_model *= factor


    plt.plot(smoothed_wl, smoothed_spec_model, label='Binned Planet Model', color='orange', linewidth=0.5)
    plt.plot(smoothed_wl, smoothed_spec_subtract, label='Binned Subtracted Spectrum', color='green', alpha=0.7, linewidth=0.5)

plt.xlabel('Wavelength (m)')
plt.ylabel('Flux (counts)')
plt.title('Binned Fitted Planet Spectrum')
plt.legend()
plt.show()


#smoothen the spectrum with a low resolution filter to make the absorption lines more visible
def smooth_spectrum(spectrum, window_length=151, polyorder=2):
    """Smooth the spectrum using a Savitzky-Golay filter."""
    spectrum = np.nan_to_num(spectrum, nan=0.0)
    return savgol_filter(spectrum, window_length=window_length, polyorder=polyorder)

plt.figure(figsize=(12,5))
for det in range(3):
    planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])
    planet_model_spec -= savgol_filter(planet_model_spec, window_length=300, polyorder=2)
    
    smoothed_spec_model = smooth_spectrum(planet_model_spec)

    subtract_spec = (subtract[det, order,spatial_pix]/np.nanmax(subtract[det, order,spatial_pix]))
    
    smoothed_spec_subtract = smooth_spectrum(subtract_spec)

    #factor = np.median(smoothed_spec_subtract)/np.median(smoothed_spec_model)

    #smoothed_spec_model *= factor

    plt.plot(wave_data[det,order], smoothed_spec_subtract, label='Binned Subtracted Spectrum', color='green', alpha=0.7, linewidth=2.5)
    plt.plot(wave_data[det,order], smoothed_spec_model, label='Binned Planet Model', color='orange', linewidth=1.5)

plt.xlabel('Wavelength (m)')
plt.ylabel('Flux (counts)')
plt.title('Smoothed Fitted Planet Spectrum')
plt.legend()
plt.show()

In [ ]:
img = subtract[det, order, 0:101]


wl = np.array(wave_range[0][order])

plt.imshow(subtract[det, order, 0:101, :], aspect='auto',norm=SymLogNorm(0.01, 1), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.show()

plt.imshow(fit_matrix_planet[det, order, 0:101, :], aspect='auto',norm=SymLogNorm(0.1, 1), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.show()

In [ ]:
#plt.plot(stack[-1, 18, 2048*2:2048*3])
order=1
det=0


plt.imshow(fit_matrix_star[det, order], aspect='auto',norm=SymLogNorm(0.01, 2), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.title('Stellar-fit matrix at Order %s'%(23+order))
plt.legend()
plt.show()


plt.plot(fit_matrix_star[det,order,60])
plt.show()

subtract=stack-fit_matrix_star
mask_sub=np.isfinite(subtract)
subtract[~mask_sub]=np.nan
fit_matrix_planet[~mask_sub]=np.nan

plt.imshow(subtract[det, order], aspect='auto',norm=SymLogNorm(0.01, 1), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.title('Mock Observation - Stellar-fit matrix at Order %s'%(23+order))
plt.show()

plt.imshow(fit_matrix_planet[det, order], aspect='auto',norm=SymLogNorm(0.01, 1), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.title(' Planet-fit matrix at Order %s'%(23+order))


plt.show()

plt.plot(subtract[det, order, :, 1600])
plt.show()


##### Cross-correlation

In [ ]:
ccf_model_w, ccf_model_f = np.loadtxt(prep_planet.path+'/spectra_input/planet_%s_%s_%s_%s.dat' % (prep_planet.T, prep_planet.G, prep_planet.Z, 0))

ccf_model_f -= savgol_filter(ccf_model_f, 300, 2)
#ccf_model_f -= np.nanmedian(ccf_model_f)

process_cc_A=processor_cross_correlation(wMod=ccf_model_w*1e9, fMod=ccf_model_f, wlen=wave_data, cube=subtract[:,:,0:101,:], nOrder=5, nDet=3, n_spatial=101)

Ncc=500
dRV=0.5
RVlag=(np.arange(Ncc) - (Ncc-1)//2) * dRV

ccf_sum, ccf_snr =  process_cc_A.ccf_tot(rvlag=RVlag, ncc=Ncc, v_sys=(13.95+16.7), clean_grids=[[0, 10], [-10, -1]], spatial_pix=20, central_pix=60)

In [ ]:
[plt.plot(RVlag, ccf_snr[i,:], label='%s'%i) for i in range(20-5, 20+5)]
#plt.plot(RVlag, ccf_snr[100,:])
plt.legend()
plt.show()

#### 2.5.2 Fit pair B-A

In [ ]:
v=31
planet_model=np.load(prep_planet.path+'/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))
window=500
polyorders = 2

fit_matrix_planet, fit_matrix_star, fit_alpha_star, fit_alpha_planet, fit_model, popt_list=fit_workflow_B.multiplicative_factor_fit(planet_model[:, 0:5], window_len=window,
                                                                                                                                     Polyorder=polyorders, 
                                                                                                                                     master=master_matrix_B[:, 0:5])

In [ ]:
# run several tests to check the results
po=60
i=0
x=1

stack = fit_workflow_B.spectra

#multi_factor=(fit_alpha_star[-3,:, 0:2046].T*popt_list[-3, :, 0, 0])

#plt.plot(multi_factor[1000],label=r'$c_s\alpha (\lambda)$')
plt.plot(fit_alpha_star[x,i,:,500], label=r'$ \alpha $')
#plt.vlines(x=17, ymin=0, ymax=3e-2, ls='--')
#plt.vlines(x=27, ymin=0, ymax=3e-2, ls='--')
#plt.ylim(0.647,0.649)
#plt.xlim(1000,1010)
plt.legend()
plt.show()


plt.plot(popt_list[x, i, :, 1], label=['cp'])
#plt.hlines(y=1, xmin=0, xmax=35, ls='--', colors='gray')
#plt.vlines(x=(60+po), ymin=0, ymax=0.02, ls='--')
plt.xlabel('Spatial axis')
plt.ylabel('Fitted coefficient value')
#plt.yscale('symlog')
plt.legend()
plt.show()

mask=np.array(master_matrix_B[x][i] != 0)
test_master=savgol_filter(master_matrix_B[x][i][mask], window, polyorder=2)
plt.plot(master_matrix_B[x][i], alpha=0.3)
plt.plot(test_master, label='lowpass')

plt.legend()
plt.show()

plt.plot(fit_alpha_star[x, i, 60])
#plt.ylim(0.64, 0.66)

In [ ]:
ccf_model_w, ccf_model_f = np.loadtxt(prep_planet.path+'/spectra_input/planet_%s_%s_%s_%s.dat' % (prep_planet.T, prep_planet.G, prep_planet.Z, 0))

ccf_model_f -= savgol_filter(ccf_model_f, 500, 2)
ccf_model_f -= np.nanmedian(ccf_model_f)

subtract = fit_workflow_B.spectra - fit_matrix_star

process_cc_B=processor_cross_correlation(wMod=ccf_model_w*1e9, fMod=ccf_model_f, wlen=wave_data, cube=subtract[:,:,0:81,:], nOrder=5, nDet=3, n_spatial=71)

Ncc=500
dRV=1.0
RVlag=(np.arange(Ncc) - (Ncc-1)//2) * dRV

ccf_sum, ccf_snr =  process_cc_B.ccf_tot(rvlag=RVlag, ncc=Ncc, v_sys=(13.95+16.7), clean_grids=[[0, 10], [-10, -1]], spatial_pix=(20), central_pix=(60))

### 2.6 Spectral Fitting with more Stellar Components

In [ ]:
import numpy as np
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.linalg import solve

import scipy.ndimage
import scipy.constants

from scipy import interpolate
from astropy.io import fits
from PyAstronomy import pyasl
from exotic_ld import StellarLimbDarkening

import matplotlib.pyplot as plt
import tqdm

import glob




import os

class Spectra_Preparator:
    def __init__(self, path, folder_name, T, G, Z):
        self.path = path
        self.folder_name = folder_name
        self.path_a = os.path.join(path, folder_name)
        self.T = T
        self.G = G
        self.Z = Z
        self.ld_coefficients = None

    def generate_synthetic_stellar_models(self, model_spec, n_mus, factor):
        # Generate I(lambda, mu).
        n_wvs = len(model_spec)
        mus = np.linspace(0, 1, n_mus)
        stellar_intensity = []

        for mu in mus:
            stellar_intensity.append(model_spec*factor)

        return mus, np.array(stellar_intensity).T

    def compute_limb_darkening(self, wavelength_range, throughput_files, Mode='phoenix', flux=None, wave=None):
        """
        Compute limb darkening coefficients for different bands.
        """
        ld_coeffs = np.zeros((len(wavelength_range), 1))

        for idx, (lam_range, throughput_file) in enumerate(zip(wavelength_range, throughput_files)):
            throughput = np.loadtxt(throughput_file)
            wave_throughput, frac_throughput = throughput[:, 0], throughput[:, 1]

            if Mode == 'phoenix':
                sld = StellarLimbDarkening(
                    ld_model=Mode,
                    ld_data_path=os.path.join(self.path, 'ld_data'),
                    M_H=self.Z,
                    Teff=self.T,
                    logg=self.G
                )

                cs = sld.compute_linear_ld_coeffs(
                    mode='custom',
                    wavelength_range=[lam_range[0][0]*10, lam_range[-1][-1]*10],
                    custom_wavelengths=wave_throughput,
                    custom_throughput=frac_throughput
                )

            elif Mode =='custom':
                mus, intensities = self.generate_synthetic_stellar_models(flux, 70, factor=1e-8)
                
                sld=StellarLimbDarkening(
                    ld_model=Mode, 
                    ld_data_path=os.path.join(self.path, 'ld_data'), \
                    custom_wavelengths=wave, 
                    custom_stellar_model=intensities, 
                    custom_mus=mus
                )
                


                cs=sld.compute_linear_ld_coeffs(
                    mode='custom', wavelength_range=[lam_range[0][0]*10, lam_range[-1][-1]*10],   
                    custom_wavelengths=wave_throughput, 
                    custom_throughput=frac_throughput
                )




            ld_coeffs[idx] = np.float16(cs)

        self.ld_coefficients = ld_coeffs
        return ld_coeffs

    def load_phoenix_spectrum(self):
        """
        Load high resolution PHOENIX model spectrum.
        """
        flux = fits.open(os.path.join(
            self.path_a,
            f"lte0{self.T}-{self.G}0{self.Z}.PHOENIX-ACES-AGSS-COND-2011-HiRes.fits"
        ))[0].data

        wave = fits.open(os.path.join(
            self.path_a,
            "WAVE_PHOENIX-ACES-AGSS-COND-2011.fits"
        ))[0].data

        return wave, flux

    def gaussian_broaden(self, wavelength, flux, R):
        log_wavelength = np.log(wavelength)
        fwhm_log = 1 / R
        sigma_log = fwhm_log / 2.355
        broadened_flux = scipy.ndimage.gaussian_filter1d(
            flux,
            sigma=sigma_log / (log_wavelength[1] - log_wavelength[0])
        )
        return broadened_flux

    def resolution_sample(self, ave_R, wavelength):
        lam_range = np.max(wavelength) - np.min(wavelength)
        lam_mid = (np.max(wavelength) + np.min(wavelength)) / 2
        N = ave_R * (lam_range / lam_mid) + 1
        return int(N)

    def apply_broadening(self, wave, flux, lam_range, ld_coeff, R=1e5, rotation=5e0, reu=500, step_size=1e-11):

        if (wave[1]-wave[0]) != (wave[-1]-wave[-2]):

            print ('Sample grid is not equal. Use pyasl.equidistantInterpolation to produce equidistantly sampled data. ')

            
            wave_new, flux_new = pyasl.equidistantInterpolation(wave, flux, step_size)

    
        mask =  ( (wave_new >= (lam_range[0][0]-reu)*1e-9) & (wave_new <= (lam_range[-1][-1]+reu)*1e-9) )
        rflux = pyasl.fastRotBroad(
            wvl=wave_new[mask],
            flux=flux_new[mask],
            epsilon=ld_coeff,
            vsini=rotation
        )
        broadened_flux = self.gaussian_broaden(wave_new[mask], rflux, R)

        N = self.resolution_sample(R*2, wave_new[mask])
        wave_grid = np.linspace(np.min(wave_new[mask]), np.max(wave_new[mask]), N)
        inter_flux = np.interp(wave_grid, wave_new[mask], broadened_flux)

        return wave_grid, inter_flux


    def radial_velocity_shift(self, wave, flux, velocity):
        """
        Apply radial velocity shift in km/s.
        """
        shifted_wave = wave * (1 + (velocity * 1e3 / scipy.constants.c))
        return shifted_wave, flux
    
    def spectrum_cutter(self, lam_range, wave, flux, saving_file=True, saving_path = None, filenames = None, ignore_detector=True, Wave_solution=None):
        
        if ignore_detector == True:
            print ('The division into 3 detectros is ignored. The spectrum is only cut into different spectral orders.')
            n=0
            for i in lam_range:

                mask = ( ( wave >= i[0]*1e-9) & (wave <= i[1]*1e-9) )

                wave_cut = wave[mask]
                flux_cut = flux[mask]

                if saving_file == True:
                
                    filename = saving_path + filenames[n]

                    self.save_spectrum(filename, wave_cut, flux_cut)

                    n+=1
            
            print ('All spectra are saved in %s'%saving_path)

        if ignore_detector == False:

            spec_matrix = self.model_matrix_intepolater(wave, flux, Wave_solution)

            if saving_file == True:
                
                filename = '%s'%saving_path + '%s'%filenames

                np.save(filename, spec_matrix)

            print ('The spectral matrix is saved in %s with the shape of %s'%(saving_path, spec_matrix.shape))

            return (spec_matrix)


    def model_matrix_intepolater (self, wave_model, flux_model, wave_solution):

        spec_interp_matrix = np.zeros( shape=wave_solution.shape )

        for det in range(wave_solution.shape[0]):

            for order in range(wave_solution.shape[1]):

                wave_cut = wave_solution[det][order]

                model_mask = np.array((wave_model>=wave_cut.min())&(wave_model<=wave_cut.max()))

                spec_interp = np.interp(wave_cut, wave_model[model_mask], flux_model[model_mask])

            
                spec_interp_matrix[det][order]=spec_interp
        
        return (spec_interp_matrix)




    
    def save_spectrum(self, filename, wave, flux):
        spec = np.array([wave, flux])
        np.savetxt(filename, spec)

        return (spec)





class Spectra_fitting:
    """
    Assumes self.spectra has shape (nDet, nOrder, n_spatial, nPix)
    """

    def __init__(self, spectra2D, transmission_matrix=None, window_length=None, polyorder=None, detector_number=None):
        self.spectra = spectra2D
        self.transmission_matrix = transmission_matrix
        self.window_length = window_length
        self.polyorder = polyorder
        self.detector_number = detector_number

    def Mask_spec(self, atm_threshold, sigma_threshold, transmission_atmos=None, Mask_bad_pxiel=True):
        """
        transmission_atmos assumed to have shape compatible with spectra:
        e.g. (nDet, nOrder, nPix) or (nDet, nOrder, n_spatial, nPix) — here we assume (nDet, nOrder, nPix).
        Mask pixels below atm_threshold and outliers beyond sigma_threshold*(std).
        """
        nDet, nOrder, n_spatial, nPix = self.spectra.shape

        if transmission_atmos == None:
            transmission_atmos = self.transmission_matrix #Using the default transmission matrix

        # If transmission_atmos shape includes spatial axis, adapt:
        # Common use-case: transmission has shape (nDet, nOrder, nPix)
        # We'll treat transmission_atmos[iDet, iOrder, :] as the transmission per pixel.
        for iDet in range(nDet):
            for iOrder in range(nOrder):
                trans = transmission_atmos[iDet, iOrder]  # shape (nPix,) ideally

                # Build atmospheric mask for pixels below threshold
                low_atmo_mask = (trans <= atm_threshold)

                # For each spatial pixel compute mean/std across pixels or across spatial axis?
                # I assume we want to flag bad pixels per spatial row: find mean and std across wavelength axis for each pixel
                # mean over spatial axis -> shape (nPix,)
                if Mask_bad_pxiel == True:
                    mean_flux = np.nanmean(self.spectra[iDet, iOrder, :, :], axis=1)
                    std_flux = np.nanstd(self.spectra[iDet, iOrder, :, :], axis=1)

                    # For each spatial position, mask outliers along pixel axis
                    for isp in range(n_spatial):
                        flux_row = self.spectra[iDet, iOrder, isp, :]
                        # bad pixels where abs(flux - mean) > sigma_threshold * std
                        bad_pixel_mask = np.abs(flux_row - mean_flux[isp]) > (sigma_threshold * (std_flux[isp] + 1e-12))
                        # combine with atmosphere mask
                        combined_mask = bad_pixel_mask | low_atmo_mask
                        flux_row[combined_mask] = np.nan
                        self.spectra[iDet, iOrder, isp, :] = flux_row
                else:
                    for isp in range(n_spatial):
                        flux_row = self.spectra[iDet, iOrder, isp, :]
                        flux_row[low_atmo_mask] = np.nan
                        self.spectra[iDet, iOrder, isp, :] = flux_row

        return self.spectra

    def stellar_master_spec(self, wave, cut_off, center=None):
        """
        Compute a master stellar spectrum by integrating spatial pixels around center.
        By default it uses detector=det and order=order to find the spatial center.
        wave is the wavelength array matching last axis (length nPix).
        """
        nDet, nOrder, n_spatial, nPix = self.spectra.shape
        spec_master_matrix = np.zeros(shape=wave.shape)

        for iDet in range(nDet):
            for iOrder in range(nOrder):
                if center is None:
                    # integrate flux across wavelength for each spatial pixel to find center
                    # mask=np.isfinite(self.spectra[iDet, iOrder])

                    spatial_curve = np.trapz(np.nan_to_num(self.spectra[iDet, iOrder], nan=0.0), x=wave[iDet,iOrder], axis=1)  # shape (n_spatial,)
                    
                    if np.all(np.isnan(spatial_curve)):
                        print(f"Warning: all NaN curve in Det {iDet}, Order {iOrder}")
                        continue

                    center = int(np.nanargmax(spatial_curve))

                    print('Center: pixel %s' % center)

                i0 = max(0, center - cut_off)
                i1 = min(n_spatial, center + cut_off + 1)
                spec_master_matrix[iDet, iOrder, :] = np.trapz(np.nan_to_num(self.spectra[iDet, iOrder, i0:i1, :], nan=0.0), axis=0)  # integrate over selected spatial px
        

        return spec_master_matrix

    def fit_star(self, c, master, observation, window_length, polyorder, cp=None):

        window_length=self.window_length
        polyorder=self.polyorder

        smoothed_master = savgol_filter(master, window_length, polyorder)
        smoothed_obs = savgol_filter(observation, window_length, polyorder)
        # avoid division by zero
        denom = smoothed_master.copy()
        denom[denom == 0] = np.nan
        alpha_star = smoothed_obs / denom
        stellar_real_obs = c * alpha_star * master
        stellar_model_contribution = alpha_star * master
        return (stellar_real_obs, stellar_model_contribution, alpha_star)

    def fit_planet(self, c, master, transmission, spec_planet, window_length, polyorder):

        window_length=self.window_length
        polyorder=self.polyorder

        smoothed_master = savgol_filter(master, window_length, polyorder)
        smoothed_planet = savgol_filter(transmission * spec_planet, window_length, polyorder)
        denom = smoothed_master.copy()
        denom[denom == 0] = np.nan
        alpha_planet = c * smoothed_planet / denom
        planet_obs = c * transmission * spec_planet
        planet_leakage = c * master * alpha_planet
        planet_real_obs = planet_obs - planet_leakage
        planet_model_contribution = transmission * spec_planet - alpha_planet * master
        return (planet_real_obs, planet_model_contribution, planet_obs, planet_leakage, alpha_planet)


    def _shift_spectrum(self, master, shift_pix):
        """
        Shift 1D spectrum by shift_pix (can be float) using linear interpolation.
        master: 1D array length nPix
        shift_pix: shift in pixels (positive shifts spectrum to the right -> values move to higher index)
        Returns an array same length as master with NaNs where interpolation falls outside.
        """
        n = master.shape[0]
        x = np.arange(n)
        x_src = x - shift_pix  # map target indices -> source indices
        # np.interp cannot produce NaN, so we handle outside-range manually
        shifted = np.interp(x, x_src, master, left=np.nan, right=np.nan)
        return shifted


    # --- keep all your previous methods unchanged: Mask_spec, stellar_master_spec, fit_star, fit_planet, etc. ---
    def multiplicative_factor_fit(self,
                              planet_model_spectrum,
                              master=None,
                              window_len=None,
                              Polyorder=None,
                              shifts_pixels=None,
                              expected_planet_position=None,
                              mask_radius=None):
        """
        Extended version with multiple shifted stellar spectra.

        Arguments
        ----------
        planet_model_spectrum : (nDet, nOrder, nPix)
            Planet model spectrum per detector/order.
        master : (nDet, nOrder, nPix)
            Stellar master spectrum (before shifting).
        shifts_pixels : list or np.array
            Pixel shifts for stellar components.

        Returns
        -------
        fit_matrix_planet, fit_matrix_star, fit_alpha_star, fit_alpha_planet, fit_model, coeffs
        """
        self.window_length = window_len
        self.polyorder = Polyorder

        if shifts_pixels is None:
            shifts_pixels = [0.0]
        shifts_pixels = np.asarray(shifts_pixels, dtype=float)
        n_stellar = len(shifts_pixels)

        nDet, nOrder, n_spatial, nPix = self.spectra.shape

        coeffs = np.zeros((nDet, nOrder, n_spatial, n_stellar + 1))
        fit_matrix_planet = np.zeros_like(self.spectra)
        fit_matrix_star = np.zeros_like(self.spectra)
        fit_alpha_star = np.zeros_like(self.spectra)
        fit_alpha_planet = np.zeros_like(self.spectra)
        fit_model = np.zeros((n_stellar + 1, nDet, nOrder, n_spatial, nPix))

        for iDet in range(nDet):
            for iOrder in range(nOrder):
                # Select planet model slice
                planet_model = planet_model_spectrum[iDet, iOrder, :]

                # Select master for this detector/order
                master_vec = master[iDet, iOrder, :] if master.ndim == 3 else master
                mask_master = np.isfinite(master_vec) & (master_vec != 0)

                # Prepare shifted masters
                shifted_masters = []
                for shift in shifts_pixels:
                    shifted = self._shift_spectrum(master_vec, shift)
                    shifted[np.isnan(shifted)] = 0.0
                    shifted_masters.append(shifted)
                shifted_masters = np.array(shifted_masters)

                for isp in range(n_spatial):
                    obs = self.spectra[iDet, iOrder, isp, :]
                    mask_obs = np.isfinite(obs)
                    mask = mask_master & mask_obs

                    if np.sum(mask) < 10:
                        coeffs[iDet, iOrder, isp, :] = np.nan
                        continue

                    # Construct design matrix columns: each shifted stellar component + planet
                    A_cols = []
                    alpha_star_components = []
                    for k in range(n_stellar):
                        _, star_model_contrib, alpha_star = self.fit_star(
                            c=1.0,
                            master=shifted_masters[k][mask],
                            observation=obs[mask],
                            window_length=self.window_length,
                            polyorder=self.polyorder
                        )
                        A_cols.append(star_model_contrib)
                        alpha_star_components.append(alpha_star)

                    # Planet column
                    _, planet_model_contrib, _, _, alpha_planet = self.fit_planet(
                        c=1.0,
                        master=master_vec[mask],
                        transmission=self.transmission_matrix[iDet, iOrder][mask],
                        spec_planet=planet_model[mask],
                        window_length=self.window_length,
                        polyorder=self.polyorder
                    )

                    A_cols.append(planet_model_contrib)

                    # print correlations for diagnostics (optional)
                    for cols in A_cols[:-1]:
                        try:
                            corr = np.corrcoef(cols, planet_model_contrib)[0, 1]
                        except Exception:
                            corr = np.nan
                        # you can comment prints in production
                        if np.isfinite(corr) and abs(corr) > 0.9:
                            print(f"[Det{ iDet },Ord{ iOrder },Row{ isp }] High corr star vs planet: {corr:.3f}")


                    
                    A = np.vstack(A_cols).T

                    # Solve linear least squares A·coeff = obs
                    # Black the rows close to the star from cp fitting while only use cs fitting if needed
                    
                    star_center = n_spatial // 2 if self.detector_number is None else self.detector_number[iDet]
                    planet_center = expected_planet_position if expected_planet_position is not None else star_center
                    
                    # pixels around star center to mask planet fitting
                    if mask_radius is not None and abs(isp - planet_center) >= mask_radius:
                        # zero last column (planet) to prevent cp fitting near star
                        A[:, -1] = 0.0
                    else:

                        pass
                    

                    y = obs[mask]
                    try:
                        sol, *_ = np.linalg.lstsq(A, y, rcond=None)
                    except Exception:
                        sol = np.full(A.shape[1], np.nan)
                    coeffs[iDet, iOrder, isp, :] = sol

                    # Rebuild fitted star and planet matrices

                    # Planet
                    _, planet_model_contrib, planet_obs, planet_leakage, alpha_planet = self.fit_planet(
                        c=sol[-1],
                        master=master_vec[mask],
                        transmission=self.transmission_matrix[iDet, iOrder][mask],
                        spec_planet=planet_model[mask],
                        window_length=self.window_length,
                        polyorder=self.polyorder
                    )
                    fit_matrix_planet[iDet, iOrder, isp, :][mask] = planet_obs
                    fit_model[-1, iDet, iOrder, isp, :][mask] = planet_model_contrib
                    fit_alpha_planet[iDet, iOrder, isp, :][mask] = alpha_planet


                    # Star (sum of shifted components)
                    star_fit_total = np.zeros(nPix)
                    #alpha_star_sum = 0.0
                    for k in range(n_stellar):
                        star_obs, star_model_contrib, alpha_star = self.fit_star(
                            c=sol[k],
                            master=shifted_masters[k][mask],
                            observation=obs[mask],
                            window_length=self.window_length,
                            polyorder=self.polyorder
                        )

                        fit_matrix_star[iDet, iOrder, isp, :][mask] += star_obs - (planet_leakage/n_stellar)
                        fit_model[k, iDet, iOrder, isp, :][mask] = star_model_contrib
                        if np.isclose(shifts_pixels[k], 0.0):
                        
                        #alpha_star_sum += alpha_star
                            fit_alpha_star[iDet, iOrder, isp, :][mask] = alpha_star - alpha_planet                   

                                       

        return (fit_matrix_planet,
                fit_matrix_star,
                fit_alpha_star,
                fit_alpha_planet,
                fit_model,
                coeffs)



    


class processor_cross_correlation:
    def __init__(self, wMod, fMod, wlen, cube, nOrder, n_spatial, nDet):
        """
        wMod, fMod : 1D model wavelength and flux (same length)
        wlen : observed wavelength grid with shape (nDet, nOrder, nPix)
        cube : observed cube shape (nDet, nOrder, n_spatial, nPix)
        """
        self.wMod = wMod
        self.fMod = fMod
        self.wlen = wlen
        self.cube = cube
        self.n_spatial = n_spatial
        self.nOrder = nOrder
        self.nDet = nDet

    def xcorr(self, f, g):
        """
        Normalized cross-correlation between 1D arrays f and g (no in-place edits).
        Returns NaN if varf or varg is zero.
        """
        f = np.asarray(f, dtype=float)
        g = np.asarray(g, dtype=float)
        nx = len(f)
        if nx == 0:
            return np.nan
        I = np.ones(nx)
        f_mean_sub = f - (np.dot(f, I) / nx)
        g_mean_sub = g - (np.dot(g, I) / nx)
        R = np.dot(f_mean_sub, g_mean_sub) / nx
        varf = np.dot(f_mean_sub, f_mean_sub) / nx
        varg = np.dot(g_mean_sub, g_mean_sub) / nx
        denom = np.sqrt(varf * varg)
        if denom == 0 or np.isnan(denom):
            return np.nan
        return R / denom

    def get_cc_grid(self, rvlag, ncc):
        """
        Compute CCF cube with shape (nDet, nOrder, n_spatial, ncc)
        """
        ccf = np.zeros((self.nDet, self.nOrder, self.n_spatial, ncc))
        coef_spline = interpolate.splrep(self.wMod, self.fMod, s=0.0)

        for irv, rv in enumerate(rvlag):
            beta = rv / 2.998e5
            # Relativistic shift: For each observed wavelength grid
            # wShift will be same shape as self.wlen (nDet,nOrder,nPix)
            wShift = self.wlen * np.sqrt((1.0 - beta) / (1.0 + beta))
            # Evaluate model flux at the shifted observed wavelengths
            # splev supports array input and will return an array same shape as wShift
            intMod = interpolate.splev(wShift, coef_spline, der=0)  # shape (nDet,nOrder,nPix)

            for iDet in range(self.nDet):
                for iOrder in range(self.nOrder):
                    for iObs in range(self.n_spatial):
                        obs = self.cube[iDet, iOrder, iObs, :]
                        model_row = intMod[iDet, iOrder, :]
                        # mask NaNs to avoid propagating
                        mask = np.isfinite(obs) & np.isfinite(model_row)
                        if np.sum(mask) < 5:
                            ccf[iDet, iOrder, iObs, irv] = np.nan
                        else:
                            ccf[iDet, iOrder, iObs, irv] = self.xcorr(obs[mask], model_row[mask])

        self.rvlag = rvlag
        self.ncc = ncc
        return ccf

    def ccf_tot(self, rvlag, ncc, plot=True, subtract_continuum='med_flux', normalization='median subtracted',
                v_sys=None, clean_grids=None, po=None, central_pix=None, spatial_pix=None):
        """
        Returns ccf_Sum (shape (n_spatial, ncc) summed across dets and orders),
                ccf_SNR (same shape)
        """

        # compute medFlux per det,order,pix by taking median over spatial axis
        medFlux = np.zeros((self.nDet, self.nOrder, self.wlen.shape[-1]))
        for iDet in range(self.nDet):
            for iOrder in range(self.nOrder):
                # cube slice shape (n_spatial, nPix), median over spatial axis -> (nPix,)
                medFlux[iDet, iOrder, :] = np.nanmedian(self.cube[iDet, iOrder, :, :], axis=0)

        # ccfWeight could be sum across dets then across orders if desired; keep it optional
        ccfWeight = np.sum(medFlux, axis=(0, 1))  # shape (nPix,)
        if np.nansum(ccfWeight) != 0:
            ccfWeight = ccfWeight / np.nansum(ccfWeight)

        ccf = self.get_cc_grid(rvlag, ncc)  # shape (nDet, nOrder, n_spatial, ncc)

        # Sum across detectors and orders to get per spatial x rv matrix
        ccf_Sum = np.nansum(ccf, axis=(0, 1))  # shape (n_spatial, ncc)

        # Normalization options
        if normalization == 'median':
            denom = np.nanmedian(ccf_Sum)
            if denom != 0:
                ccf_Sum = (ccf_Sum - denom) / denom
        elif normalization == 'max':
            mval = np.nanmax(ccf_Sum)
            if mval != 0:
                ccf_Sum = ccf_Sum / mval
        elif normalization == 'median subtracted':
            for iObs in range(self.n_spatial):
                med = np.nanmedian(ccf_Sum[iObs, :])
                ccf_Sum[iObs, :] -= med

        # optional plotting
        if plot:
            try:
                plotMatrix.plotMatrix(ccf_Sum, rvlag, np.arange(0, self.n_spatial, 1),
                                      'Radial velocity (km/s)', 'spatial axis', stretch=True,
                                      planet_posi=spatial_pix, scale='log')
            except Exception:
                plt.imshow(ccf_Sum, aspect='auto', origin='lower', extent=(rvlag[0], rvlag[-1], 0, self.n_spatial))
                plt.colorbar()
            if central_pix is not None:
                plt.hlines(y=central_pix, xmin=rvlag[0], xmax=rvlag[-1], ls='--', colors='lightgray')
            if v_sys is not None:
                plt.vlines(x=v_sys, ymin=0, ymax=(self.n_spatial - 1), ls='--', colors='lightgray')
            plt.show()

        # compute SNR by building a 'clean' grid (exclude RV ranges where signal sits)
        if clean_grids is None:
            # fallback: compute std across entire rv axis
            std_ccf = np.nanstd(ccf_Sum)
        else:
            # clean_grids expected like [(rv_idx0, rv_idx1), (rv_idx2, rv_idx3)]
            g0, g1 = int(clean_grids[0][0]), int(clean_grids[0][1])
            g2, g3 = int(clean_grids[1][0]), int(clean_grids[1][1])
            ccf_clean_map = np.concatenate((ccf_Sum[:, g0:g1], ccf_Sum[:, g2:g3]), axis=1)
            std_ccf = np.nanstd(ccf_clean_map)

        ccf_SNR = np.zeros_like(ccf_Sum)
        if std_ccf == 0:
            ccf_SNR[:] = np.nan
        else:
            ccf_SNR = ccf_Sum / std_ccf

        if plot:
            try:
                plotMatrix.plotMatrix(ccf_SNR, rvlag, np.arange(0, self.n_spatial, 1),
                                      'Radial velocity (km/s)', 'spatial axis', stretch=True,
                                      planet_posi=spatial_pix)
            except Exception:
                plt.imshow(ccf_SNR, aspect='auto', origin='lower', extent=(rvlag[0], rvlag[-1], 0, self.n_spatial))
                plt.colorbar()
            if central_pix is not None:
                plt.hlines(y=central_pix, xmin=rvlag[0], xmax=rvlag[-1], ls='--', colors='lightgray')
            if v_sys is not None:
                plt.vlines(x=v_sys, ymin=0, ymax=(self.n_spatial - 1), ls='--', colors='lightgray')
            plt.title('Cross-correlated SNR')
            plt.show()

        return (ccf_Sum, ccf_SNR)



class processor_likelihood_map:
    def __init__(self):
        pass

    def covariance_calculator(self, Fit_Matrix, ABBA_series, N_exposures):
        fit_matrix_single = Fit_Matrix / N_exposures
        fit_matrix_ex = np.expand_dims(fit_matrix_single, axis=1)
        fit_matrix_ex = np.repeat(fit_matrix_ex, repeats=N_exposures, axis=1)
        resi = ABBA_series - fit_matrix_ex
        var_resi = np.var(resi, axis=1)
        return (var_resi, resi)

    def likelihood_calculator(self, d=None, M=None, Sigma_0_flat=None, log_sigma_0=True, prior_psi=1.0, gamma=2):
        Sigma_0 = np.diagflat(Sigma_0_flat)
        # avoid log of zero
        safe_sigma_flat = np.array(Sigma_0_flat) + 1e-12
        log_Sigma_0 = np.diagflat(np.log(safe_sigma_flat))
        Sigma_0_inv = np.linalg.inv(Sigma_0)

        # Matrix arithmetic
        MT_Sigma0_inv_M = M.T @ Sigma_0_inv @ M
        MT_Sigma0_inv_d = d.T @ Sigma_0_inv @ M

        # Solve for c_hat: shape (Nc,)
        try:
            c_hat_T = solve(MT_Sigma0_inv_M, MT_Sigma0_inv_d)
            c_hat = c_hat_T.T
        except Exception as e:
            # fallback if singular
            c_hat = np.linalg.lstsq(MT_Sigma0_inv_M, MT_Sigma0_inv_d, rcond=None)[0].T

        residual = d - M @ c_hat
        chi_0_squared = float(residual.T @ Sigma_0_inv @ residual)

        # determinant handling (use slogdet for stability)
        sign_det, logdet = np.linalg.slogdet(MT_Sigma0_inv_M)
        if sign_det <= 0:
            # singular or negative determinant; fall back to small value
            logdet = np.log(np.abs(np.linalg.det(MT_Sigma0_inv_M) + 1e-30))

        Nd = len(d)
        Nc = M.shape[1]

        if not log_sigma_0:
            det_sigma0 = np.linalg.det(Sigma_0)
            log_likelihood = (np.log(prior_psi) - 0.5 * np.log(det_sigma0 * max(np.linalg.det(MT_Sigma0_inv_M), 1e-30))
                              + ((Nd - Nc + gamma - 1) / 2) * np.log(1.0 / max(chi_0_squared, 1e-30)))
        else:
            sum_log_sigma0 = np.sum(np.log(safe_sigma_flat))
            log_likelihood = (np.log(prior_psi) - 0.5 * (sum_log_sigma0 - logdet)
                              + ((Nd - Nc + gamma - 1) / 2) * np.log(1.0 / max(chi_0_squared, 1e-30)))
        return log_likelihood

    def likelihood_map(self, observation_matrix, model_matrix, n_vgrids, sigma_matrix, prior, matrix_components=2, log_sigma=True):
        """
        observation_matrix: shape (nObs, nPix)
        model_matrix: either shape (nVgrid, n_components, nObs, nPix) or (nVgrid, nObs, n_components, nPix)
        sigma_matrix: shape (nObs, nPix) variances
        Returns log_likelihood_mapp shape (nObs, nVgrid)
        """
        map_size = (observation_matrix.shape[0], n_vgrids)
        log_likelihood_mapp = np.zeros(shape=map_size)

        for v in tqdm.tqdm(range(model_matrix.shape[0])):
            for i in range(observation_matrix.shape[0]):
                variance_flat = sigma_matrix[i].flatten()
                mask_finite = np.isfinite(observation_matrix[i])
                mask_finite_var = np.isfinite(variance_flat)

                combined_mask = mask_finite & mask_finite_var
                if np.sum(combined_mask) < 5:
                    log_likelihood_mapp[i, v] = np.nan
                    continue

                if matrix_components == 1:
                    fit_model_mask = np.atleast_2d(model_matrix[v][i][combined_mask]).T  # shape (nPix_mask, 1)
                elif matrix_components == 2:
                    # accomodate shapes; I assume model_matrix[v] returns an array with shape (2, nObs, nPix)
                    m0 = model_matrix[v][0, i, combined_mask]
                    m1 = model_matrix[v][1, i, combined_mask]
                    fit_model_mask = np.vstack((m0, m1)).T  # shape (nPix_mask, 2)
                else:
                    print("matrix_components must be 1 or 2")
                    log_likelihood_mapp[i, v] = np.nan
                    continue

                output = self.likelihood_calculator(d=observation_matrix[i][combined_mask],
                                                    M=fit_model_mask,
                                                    Sigma_0_flat=variance_flat[combined_mask],
                                                    log_sigma_0=log_sigma,
                                                    prior_psi=prior, gamma=2)
                log_likelihood_mapp[i, v] = output

        return log_likelihood_mapp


In [ ]:
fit_workflow_A = Spectra_fitting(spectra2D=flux2d_A, transmission_matrix=trans_matrix, window_length=300, polyorder=2)
spectra_masked_A=fit_workflow_A.Mask_spec(atm_threshold=0.6,  sigma_threshold=3, Mask_bad_pxiel=True)

master_matrix_A = fit_workflow_A.stellar_master_spec(wave=wave_data, cut_off=2)

master_matrix_B = fit_workflow_B.stellar_master_spec(wave=wave_data, cut_off=2)

#combine_A_B= fit_workflow_A.spectra[:,0:5]+fit_workflow_B.spectra[:,0:5]

fit_workflow_A.spectra=fit_workflow_A.spectra[:,0:5]
fit_workflow_B.spectra=fit_workflow_B.spectra[:,0:5]



In [ ]:
v= 31
planet_model=np.load(prep_planet.path+'/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))
window=500
polyorders = 2
shifts=np.arange(-3,4,1)

fit_matrix_planet, fit_matrix_star, fit_alpha_star, fit_alpha_planet, fit_model, popt_list=fit_workflow_A.multiplicative_factor_fit(planet_model[:, 0:5], window_len=window,
                                                                                                                                     Polyorder=polyorders, 
                                                                                                                                     master=master_matrix_A[:, 0:5],
                                                                                                                                     shifts_pixels=shifts,
                                                                                                                                     expected_planet_position=20,
                                                                                                                                     mask_radius=15)

In [ ]:
# run several tests to check the results
po=60
i=1
x=1

#multi_factor=(fit_alpha_star[-3,:, 0:2046].T*popt_list[-3, :, 0, 0])

#plt.plot(multi_factor[1000],label=r'$c_s\alpha (\lambda)$')
plt.plot(np.trapz(fit_alpha_star[x,i],axis=1), label=r'$ \alpha_s $')
plt.plot(np.trapz(fit_alpha_planet[x,i],axis=1), label=r'$ \alpha_{p} $')
fit_alpha_all=np.trapz(fit_alpha_star[x,i],axis=1)+np.trapz(fit_alpha_planet[x,i],axis=1)
plt.plot(fit_alpha_all, label=r'$ \alpha_{total} $')
#plt.vlines(x=17, ymin=0, ymax=3e-2, ls='--')
#plt.vlines(x=27, ymin=0, ymax=3e-2, ls='--')
plt.ylim(-1,5)
plt.yscale('symlog')
plt.xlim(5,35)
plt.ylim(0, np.nanmax(fit_alpha_all[5:35])*1.2)

plt.legend()
plt.show()


plt.plot(popt_list[x, i, :, -1], label=['cp'])
#plt.hlines(y=1, xmin=0, xmax=35, ls='--', colors='gray')
#plt.vlines(x=(60+po), ymin=0, ymax=0.02, ls='--')
plt.xlabel('Spatial axis')
plt.ylabel('Fitted coefficient value')
#plt.yscale('symlog')
plt.legend()
plt.show()

mask=np.array(master_matrix_A[x][i] != 0)
test_master=savgol_filter(master_matrix_A[x][i][mask], window, polyorder=2)
plt.plot(master_matrix_A[x][i], alpha=0.3)
plt.plot(test_master, label='lowpass')

plt.legend()
plt.show()

plt.plot(fit_alpha_star[x, i, 60])
#plt.ylim(0.64, 0.66)

In [ ]:
order=1
det=1
stack = fit_workflow_A.spectra
central_pix=60
spatial_pix=int(21)

planet_model=np.load(prep_planet.path+'/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))

planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])

#planet_model_spec -= savgol_filter (planet_model_spec, window_length=300, polyorder=2)

planet_model_spec -= np.nanmedian(planet_model_spec)

po = int (central_pix-spatial_pix)

mask_data = np.isfinite(stack[det, order, central_pix])
mask_wave = np.array(wave_data[det,order]!=0)

stack[det, order, central_pix][~mask_data] = np.nan


plt.plot(wave_data[det,order][mask_wave&mask_data], stack[det, order, central_pix][mask_wave&mask_data], label='original data')
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix_star[det, order, central_pix][mask_wave&mask_data], label='stellar fit')

plt.xlabel('wavelength (m)')
plt.ylabel('flux (counts)')
plt.title('Pix %s'%central_pix)

plt.legend()
plt.show()

plt.plot(wave_data[det,order][mask_wave&mask_data], stack[det, order,spatial_pix][mask_wave&mask_data],label='original data')
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix_star[det, order,spatial_pix][mask_wave&mask_data], label='stellar fit')


plt.title('Pix %s'%spatial_pix)

plt.xlabel('wavelength (m)')
plt.ylabel('flux (counts)')

plt.legend()
plt.show()


fit_matrix=fit_matrix_star+fit_matrix_planet

mask_fit=np.array(fit_matrix_star==0.)

fit_matrix_star[mask_fit]=np.nan

mask_fit=np.array(fit_matrix_planet==0.)

fit_matrix_planet[mask_fit]=np.nan






#plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, stack[order,spatial_pix][mask_wave&mask_data],label='original data')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_star[det, order,spatial_pix][mask_wave&mask_data], label='stellar fit')

#plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_star[order,spatial_pix][mask_wave&mask_data], label='stellar fit')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_planet[det, order,spatial_pix][mask_wave&mask_data], label='planet fit')

#plt.xlim(22500, 22520)
#plt.ylim(-1e3, 1.5e5)

plt.title(' Planet position')

plt.xlabel('wavelength ($\AA$)')
plt.ylabel('flux (counts)')

plt.yscale('symlog')

plt.legend()
plt.show()



#plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix_star[order,spatial_pix][mask_wave&mask_data], label='stellar fit')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, stack[det, order,spatial_pix][mask_wave&mask_data], label='Mock observation data')
plt.plot(wave_data[det,order][mask_wave&mask_data]*1e10, fit_matrix[det, order,spatial_pix][mask_wave&mask_data], label='stellar fit + planet-fit')

#plt.xlim(22500, 22520)
#plt.ylim(-1e3, 1.5e5)

plt.title(' Planet position')

plt.xlabel('wavelength ($\AA$)')
plt.ylabel('flux (counts)')

#plt.yscale('symlog')

plt.legend()
plt.show()

subtract=stack-fit_matrix_star

subtract_spec = (subtract[det, order,spatial_pix][mask_wave&mask_data]/np.nanmax(subtract[det, order,spatial_pix][mask_wave&mask_data]) - np.nanmedian(subtract[det, order,spatial_pix][mask_wave&mask_data]))
#subtract_spec -= savgol_filter(subtract_spec, window_length=300, polyorder=2)
subtract_spec -= np.nanmedian(subtract_spec)

fit_spec = fit_matrix_planet[det, order, spatial_pix][mask_wave&mask_data]
#fit_spec -= savgol_filter(fit_spec, window_length=300, polyorder=2)
fit_spec -= np.nanmedian(fit_spec)

plt.plot(wave_data[det,order][mask_wave&mask_data], subtract_spec, label='original data - stellar fit', linewidth=0.2)

#plt.plot(wave_data[det,order][mask_wave&mask_data],fit_spec ,label='planet fit')

plt.plot(wave_data[det,order], planet_model_spec,label='planet model')

plt.title('Pix %s'%spatial_pix)

plt.xlim(2348, 2352)

plt.xlabel('wavelength axis')
plt.ylabel('flux')

plt.legend()
plt.show()


#plt.plot(wave_data[det,order][mask_wave&mask_data], stack[order, spatial_pix][mask_wave&mask_data], label='mock observation at planet position')
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix[det, order, spatial_pix][mask_wave&mask_data], label='planet position', alpha=0.9)
plt.plot(wave_data[det,order][mask_wave&mask_data], fit_matrix[det, order, central_pix+po][mask_wave&mask_data], label='reference position', alpha=0.7)


plt.xlabel('wavelength (m)')
plt.ylabel('flux (counts)')

plt.title('Fit Matrix $\mathbf{M_\psi}$')

plt.legend()
plt.show()



plt.plot(wave_data[det,order][mask_wave&mask_data], fit_spec, label='observation')
plt.plot(wave_data[det,order][mask_wave&mask_data], planet_model_spec[mask_wave&mask_data], label='Planet model', alpha=0.6)

plt.xlabel('wavelength axis')
plt.ylabel('flux (counts)')
plt.title("Fitted planet flux $d_p\'$")
plt.legend()

plt.show()

#-----------bin the spectrum to increase S/N ratio-----
def bin_spectrum(wavelength, spectrum, bin_size=40):
    """Bin the spectrum by averaging over specified bin size."""
    n_bins = len(spectrum) // bin_size
    binned_wavelength = np.nanmean(wavelength[:n_bins*bin_size].reshape(-1, bin_size), axis=1)
    binned_spectrum = np.nanmean(spectrum[:n_bins*bin_size].reshape(-1, bin_size), axis=1)
    return binned_wavelength, binned_spectrum

plt.figure(figsize=(12,5))
for det in range(3):

    planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])
    #planet_model_spec -= savgol_filter(planet_model_spec, window_length=300, polyorder=2)
    planet_model_spec -= np.nanmedian(planet_model_spec)
    
    smoothed_wl, smoothed_spec_model = bin_spectrum(wave_data[det,order], planet_model_spec)

    subtract_spec = (subtract[det, order,spatial_pix]/np.nanmax(subtract[det, order,spatial_pix]))
    subtract_spec -= np.nanmedian(subtract_spec)
    
    smoothed_wl, smoothed_spec_subtract = bin_spectrum(wave_data[det,order], subtract_spec)
    smoothed_spec_subtract -= np.nanmedian(smoothed_spec_subtract)

    #factor = np.median(smoothed_spec_subtract)/np.median(smoothed_spec_model)

    #smoothed_spec_model *= factor


    plt.plot(smoothed_wl, smoothed_spec_model, label='Binned Planet Model', color='orange', linewidth=1.5)
    plt.plot(smoothed_wl, smoothed_spec_subtract, label='Binned Subtracted Spectrum', color='green', alpha=0.7, linewidth=1.5)

plt.xlabel('Wavelength (m)')
plt.ylabel('Flux (counts)')
plt.title('Binned Fitted Planet Spectrum')
plt.legend()
plt.show()


#smoothen the spectrum with a low resolution filter to make the absorption lines more visible
def smooth_spectrum(spectrum, window_length=151, polyorder=2):
    """Smooth the spectrum using a Savitzky-Golay filter."""
    spectrum = np.nan_to_num(spectrum, nan=0.0)
    return savgol_filter(spectrum, window_length=window_length, polyorder=polyorder)

plt.figure(figsize=(12,5))
for det in range(3):
    planet_model_spec = planet_model[det, order]/np.nanmax(planet_model[det, order])
    planet_model_spec -= savgol_filter(planet_model_spec, window_length=300, polyorder=2)
    planet_model_spec -= np.nanmedian(planet_model_spec)
    
    smoothed_spec_model = smooth_spectrum(planet_model_spec)
    smoothed_spec_model -= np.nanmedian(smoothed_spec_model)

    subtract_spec = (subtract[det, order,spatial_pix]/np.nanmax(subtract[det, order,spatial_pix]))
    #subtract_spec -= savgol_filter(subtract_spec, window_length=300, polyorder=2)
    subtract_spec -= np.nanmedian(subtract_spec)
    
    smoothed_spec_subtract = smooth_spectrum(subtract_spec)
    smoothed_spec_subtract -= np.nanmedian(smoothed_spec_subtract)

    #factor = np.median(smoothed_spec_subtract)/np.median(smoothed_spec_model)

    #smoothed_spec_model *= factor

    plt.plot(wave_data[det,order], smoothed_spec_subtract, label='Binned Subtracted Spectrum', color='green', alpha=0.7, linewidth=2.5)
    plt.plot(wave_data[det,order], smoothed_spec_model, label='Binned Planet Model', color='orange', linewidth=1.5)

plt.xlabel('Wavelength (m)')
plt.ylabel('Flux (counts)')
plt.title('Smoothed Fitted Planet Spectrum')
plt.legend()
plt.show()

In [ ]:
img = subtract[det, order, 0:101]


wl = np.array(wave_range[0][order])

plt.imshow(subtract[det, order, 0:101, :], aspect='auto',norm=SymLogNorm(0.01, 1), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.colorbar(label='Flux (counts)')
plt.xlabel('Wavelength (m)')
plt.ylabel('Spatial axis (pixels)')
plt.title('Residuals after Stellar Fit')
plt.show()

plt.imshow(fit_matrix_planet[det, order, 0:101, :], aspect='auto',norm=SymLogNorm(0.1, 1), extent=[wl.min(), wl.max(), 0, img.shape[0]])
plt.show()

In [ ]:
ccf_model_w, ccf_model_f = np.loadtxt(prep_planet.path+'/spectra_input/planet_%s_%s_%s_%s.dat' % (prep_planet.T, prep_planet.G, prep_planet.Z, 0))

ccf_model_f -= savgol_filter(ccf_model_f, 300, 2)
#ccf_model_f -= np.nanmedian(ccf_model_f)

fit_window_low=5
fit_window_high=35
process_cc_A=processor_cross_correlation(wMod=ccf_model_w*1e9, fMod=ccf_model_f, wlen=wave_data,
                                          cube=subtract[:,:,fit_window_low:fit_window_high,:],
                                          nOrder=5, nDet=3, n_spatial=30)

Ncc=500
dRV=0.5
RVlag=(np.arange(Ncc) - (Ncc-1)//2) * dRV

ccf_sum, ccf_snr =  process_cc_A.ccf_tot(rvlag=RVlag, ncc=Ncc, v_sys=(13.95+16.7), clean_grids=[[0, 10], [-10, -1]],
                                        spatial_pix=15,
                                        central_pix=None)

In [ ]:
process_cc_A=processor_cross_correlation(wMod=ccf_model_w*1e9, fMod=ccf_model_f, wlen=wave_data,
                                          cube=subtract[:,:,0:101,:],
                                          nOrder=5, nDet=3, n_spatial=101)

Ncc=500
dRV=0.5
RVlag=(np.arange(Ncc) - (Ncc-1)//2) * dRV

ccf_sum, ccf_snr =  process_cc_A.ccf_tot(rvlag=RVlag, ncc=Ncc, v_sys=(13.95+16.7), clean_grids=[[0, 10], [-10, -1]],
                                        spatial_pix=20,
                                        central_pix=60)

In [ ]:
v= 31
planet_model=np.load(prep_planet.path+'/spectra_input/Planet_%s_%s_%s_%s.npy'%(prep_planet.T, prep_planet.G, prep_planet.Z, v))
window=500
polyorders = 2
shifts=np.arange(-3,4,1)

fit_matrix_planet, fit_matrix_star, fit_alpha_star, fit_alpha_planet, fit_model, popt_list=fit_workflow_A.multiplicative_factor_fit(planet_model[:, 0:5], window_len=window,
                                                                                                                                     Polyorder=polyorders, 
                                                                                                                                     master=master_matrix_B[:, 0:5],
                                                                                                                                     shifts_pixels=shifts,
                                                                                                                                     expected_planet_position=20,
                                                                                                                                     mask_radius=15)

In [ ]:
ccf_model_w, ccf_model_f = np.loadtxt(prep_planet.path+'/spectra_input/planet_%s_%s_%s_%s.dat' % (prep_planet.T, prep_planet.G, prep_planet.Z, 0))

ccf_model_f -= savgol_filter(ccf_model_f, 300, 2)
#ccf_model_f -= np.nanmedian(ccf_model_f)

subtract=fit_workflow_B.spectra-fit_matrix_star

fit_window_low=5
fit_window_high=35
process_cc_B=processor_cross_correlation(wMod=ccf_model_w*1e9, fMod=ccf_model_f, wlen=wave_data,
                                          cube=subtract[:,:,fit_window_low:fit_window_high,:],
                                          nOrder=5, nDet=3, n_spatial=30)

Ncc=500
dRV=0.5
RVlag=(np.arange(Ncc) - (Ncc-1)//2) * dRV

ccf_sum, ccf_snr =  process_cc_B.ccf_tot(rvlag=RVlag, ncc=Ncc, v_sys=(13.95+16.7), clean_grids=[[0, 10], [-10, -1]],
                                        spatial_pix=15,
                                        central_pix=None)

In [ ]:
process_cc_B=processor_cross_correlation(wMod=ccf_model_w*1e9, fMod=ccf_model_f, wlen=wave_data,
                                          cube=subtract[:,:,0:81,:],
                                          nOrder=5, nDet=3, n_spatial=81)

Ncc=500
dRV=0.5
RVlag=(np.arange(Ncc) - (Ncc-1)//2) * dRV

ccf_sum, ccf_snr =  process_cc_B.ccf_tot(rvlag=RVlag, ncc=Ncc, v_sys=(13.95+16.7), clean_grids=[[0, 10], [-10, -1]],
                                        spatial_pix=20,
                                        central_pix=60)